# Create Knowledge Probes
This generates our knowledge probes to test for factual recall.

In [2]:
import textwrap
import sys 
sys.path.append('../..')
import utils.utils as utils
import pandas as pd

with open('../../data/arxiv/cleaned_DPO.txt', 'r') as f:
    paper = f.read()


def print_wrapped(text, width=100):
    """
    Prints the given text wrapped to a specified width for better readability in notebooks.
    This function preserves paragraph breaks.
    """
    paragraphs = text.split('\n\n')
    for para in paragraphs:
        print(textwrap.fill(para, width=width))
        print()
        
print_wrapped(paper)

\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract} While large-scale unsupervised language models (LMs) learn broad world knowledge
and some reasoning skills, achieving precise control of their behavior is difficult due to the
completely unsupervised nature of their training. Existing methods for gaining such steerability
collect human labels of the relative quality of model generations and fine-tune the unsupervised LM
to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects
the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning
to maximize this estimated reward without drifting too far from the original model. In this paper we
introduce a new parameterization of the reward model in RLHF that enables extraction of the
corresponding optimal policy in clo

In [4]:
import re
import pandas as pd

def parse_paper_structure(text):
    """Parse paper into sections, subsections and paragraphs with metadata."""
    sections = []
    
    # Split by sections first
    section_pattern = r'\\section\{([^}]+)\}'
    section_splits = re.split(section_pattern, text)
    
    current_section = "Title/Abstract"
    current_section_content = ""
    
    for i in range(len(section_splits)):
        if i == 0:
            # Content before first section
            content = section_splits[i]
            current_section_content = content
        elif i % 2 == 1:
            # This is a section title
            current_section = section_splits[i]
            continue
        else:
            # This is section content
            content = section_splits[i]
            current_section_content = content
        
        # Now split by subsections within this section
        subsection_pattern = r'\\subsection\{([^}]+)\}'
        subsection_splits = re.split(subsection_pattern, content)
        
        current_subsection = "No Subsection"
        current_subsection_content = ""
        
        for j in range(len(subsection_splits)):
            if j == 0:
                # Content before first subsection
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            elif j % 2 == 1:
                # This is a subsection title
                current_subsection = subsection_splits[j]
                continue
            else:
                # This is subsection content
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            
            # Split into paragraphs
            paragraphs = [p.strip() for p in subsection_content.split('\n\n') if p.strip()]
            
            for paragraph in paragraphs:
                sections.append({
                    'section': current_section,
                    'subsection': current_subsection,
                    'paragraph': paragraph,
                    'section_text': current_section_content,
                    'subsection_text': current_subsection_content
                })
    
    return pd.DataFrame(sections)

### [NOT USED] Generate Factual Recall Knowledge Probe

In [6]:
prompt = {}
# prompt['system'] = """Your task is to extract the key knowledge facts in a paper. 

# ##
# These claims should be stand alone claims that have meaning on their own without context, offer generalizable knowledge, and are descriptive rather than prescriptive or evaluative e.g. . For instance, the claim shouldn't be specific to a particular experiment of the paper, but a general insight that follows. List them word for word how it appears in the paper."""
# prompt['system'] = """Your task is to act as a knowledge extraction engine. From the provided text, extract standalone, factual claims that represent generalizable knowledge. The claims should be objective, descriptive, and self-contained.
# # Key Criteria For Extracting Claims

# 1.  **Objective and Factual:** Extract statements presented as facts or claims, not opinions, subjective evaluations, or superlative statements.
#     *   GOOD: "Direct Preference Optimization (DPO) is an algorithm for aligning language models with human preferences."
#     *   BAD: "We believe DPO is the most promising approach for alignment." (Subjective belief)
#     *   BAD: "DPO is the best method." (Superlative statement)

# 2.  **Self-Contained:** The claim must be fully understandable on its own, without requiring context from surrounding sentences. It should not include any pronouns or references that aren't included inthe claim.
#     *   GOOD: "The DPO loss function is derived from a mapping between policy probabilities and an implicit reward."
#     *   BAD: "This loss function is more stable." (Requires external context for "This")

# 3.  **Generalizable Knowledge, not Paper-Specific Results:** Focus on definitions, mechanisms, and core concepts. Do not extract claims about the paper's specific findings, experimental setup, or citations.
#     *   GOOD: "Reinforcement Learning from Human Feedback (RLHF) typically involves training a separate reward model on preference data."
#     *   BAD: "Our experiments show a 5% improvement on the benchmark." (Specific result)
#     *   BAD: "As shown by Smith et al. (2023), the method is effective." (Relies on a citation)
prompt['system'] = """Carefully read the provided text. Your job is to extract knowledge from the provided paper.

# Detailed Instructions
 - Specifically, I want you to identify the "pieces of knowledge" from the following text.
    - We define a piece of knowledge as an informative statement that makes sense on its own without further context. 
    - The piece of knowledge can be a single sentence, or a paragraph, or even a section of the paper. Whatever the case, it should be fully self-contained and coherent. 
    - Demarcate the piece of knowledge by marking the beginning and end of the "knowledge" in the paper with <knowledge> and </knowledge> tags.
    - Again, place the tags around the entire piece of knowledge so that it's self-contained. For instance, if there is a reference to a "reward function", the earlier formal mention of the reward function should be included in the span of the piee of knowledge.
 - Do this for all pieces of knowledge from the following text.
 - Return the entire text with this annotation."""

# 2.  **Descriptive, not Prescriptive:** The claim should describe what something *is* or *does*, not give advice or instructions.
#     *   GOOD: "DPO directly optimizes the language model policy using preference data."
#     *   BAD: "To align your model, you should use the DPO method." (Prescriptive advice)

prompt['user'] = f"""Paper: {paper}"""
extracted_claims = []
import concurrent.futures

def query_single():
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0.1, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single) for _ in range(1)]
    extracted_claims = [future.result() for future in futures]
print(extracted_claims[0])

Paper: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract}
<knowledge>
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize this estimated reward without drifting too far from the original model.
In this paper we introduce a new parameterization of the reward model in RLHF that enables extraction of the corresponding op

In [20]:
extraction_prompt = """Your task is to act as a text segmenter. Carefully read the provided text from a research paper and identify all "pieces of knowledge."

# Definition of a "Piece of Knowledge"
A piece of knowledge is an informative statement that is:
1.  **Self-Contained:** It must be fully understandable on its own without needing sentences before or after it. If it uses a specific term (e.g., "the reward function"), the definition or first mention of that term should be included in the segment.
2.  **Coherent:** The extracted text should form a logical, complete thought. This can range from a single, dense sentence to a full paragraph that explains a concept.

# What to Exclude (Do NOT tag these):
- **Paper Structure & Meta-Commentary:** Sentences about the paper itself (e.g., "In this section, we describe...", "Figure 3 shows...", "Our contributions are threefold..."). 
- **Citations:** Sentences that primarily exist to cite other work (e.g., "This was previously shown by Smith et al. (2023).").
- **Transitional Sentences:** Text that only serves to connect ideas without adding new information (e.g., "Furthermore...", "Now we turn to...", "In addition to the above...").
- **Author Speculation or Rhetorical Questions:** Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?").

# Instructions
1.  Read the entire text carefully.
2.  Identify all segments that fit the definition of a "piece of knowledge" and do not fall into the exclusion categories.
3.  Demarcate each piece of knowledge by wrapping it in `<knowledge>` and `</knowledge>` tags.
4.  Return the entire original text with these annotations. Do not modify or summarize the text itself.
"""
prompt['user'] = f"""Paper: {paper}"""
extracted_claims = []
import concurrent.futures

def query_single():
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0.1, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single) for _ in range(1)]
    extracted_claims = [future.result() for future in futures]
print_wrapped(extracted_claims[0])

Paper: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract} <knowledge> While large-scale unsupervised language models (LMs) learn broad world
knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to
the completely unsupervised nature of their training. Existing methods for gaining such steerability
collect human labels of the relative quality of model generations and fine-tune the unsupervised LM
to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects
the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning
to maximize this estimated reward without drifting too far from the original model. In this paper we
introduce a new parameterization of the reward model in RLHF that enables extraction of the
corresponding op

In [21]:
extraction_prompt = """Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all "pieces of knowledge."

# Definition of a "Piece of Knowledge"
A piece of knowledge is an informative statement that is:
1.  Self-Contained: It must be fully understandable on its own without needing sentences before or after it. 
    - If it's referring to a term in a previous sentence (e.g., "the reward function"), they should be together as one unit of knowledge.
    - If the sentence is directly building on a previous sentence, either by a pronoun or a reference, they should be together as one unit of knowledge.
2.  Atomic and Coherent: It should be a single, complete thought.  
    - This can range from a single, dense sentence to a full paragraph that explains it.

# What to Exclude (Do NOT tag these):
Generally speaking, exclude sentences that mostly consist of common words and phrases that don't add much information:
- Paper Structure & Meta-Commentary: Sentences about the paper itself (e.g., "In this section, we describe...", "Figure 3 shows...", "Our contributions are threefold..."). 
- Transitional Sentences: Text that only serves to connect ideas without adding new information (e.g., "Furthermore...", "Now we turn to...", "In addition to the above...").
- Author Speculation or Rhetorical Questions: Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?"). 
- Figures and Tables: Sentences that describe figures and tables.

# Instructions
1.  Read the entire text carefully.
2.  Identify all segments that fit the definition of a "piece of knowledge" and do not fall into the exclusion categories.
3.  Demarcate each piece of knowledge by wrapping it in `<knowledge>` and `</knowledge>` tags.
4.  While you should ensure the knowledge is self-contained and atomic as much as you can, do not just place tags around an entire paragraph. Encapsulate smaller, discrete units of knowledge. When appropriate, you can also have nested tags.
5.  Return the entire original text with these annotations. Do not modify or summarize the text itself.

# Example
This is an example in which nested tags are used because the first sentence is self-contained and atomic, but the follow-up sentence is not and thus it relies on the previous sentence.
<knowledge><knowledge>In this paper we introduce a new parameterization of the reward model in RLHF that enables extraction of the corresponding optimal policy in closed form, allowing us to solve the standard RLHF problem with only a simple classification loss.</knowledge> The resulting algorithm, which we call \textit{Direct Preference Optimization} (DPO), is stable, performant, and computationally lightweight, eliminating the need for sampling from the LM during fine-tuning or performing significant hyperparameter tuning.</knowledge>
"""
# - For instance, it's okay if you have to forgo some portions of a paragraph to break it up into smaller units.
# - Citations: Sentences that primarily exist to cite other work (e.g., "This was previously shown by Smith et al. (2023).").


# Split paper into paragraphs
paper_paragraphs = paper.split('\n\n')

extracted_claims = []
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""Paper: {paragraph}"""
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, paragraph) for paragraph in paper_paragraphs]
    extracted_claims = [future.result() for future in futures]

# Print each output
for i, output in enumerate(extracted_claims, 1):
    print_wrapped(f"Paragraph {i}: {output}")
    print_wrapped("-" * 50)

Paragraph 1: Paper: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward
Model}

--------------------------------------------------

Paragraph 2: \begin{abstract} <knowledge>While large-scale unsupervised language models (LMs) learn
broad world knowledge and some reasoning skills, achieving precise control of their behavior is
difficult due to the completely unsupervised nature of their training.</knowledge>
<knowledge>Existing methods for gaining such steerability collect human labels of the relative
quality of model generations and fine-tune the unsupervised LM to align with these preferences,
often with reinforcement learning from human feedback (RLHF).</knowledge> <knowledge>However, RLHF
is a complex and often unstable procedure, first fitting a reward model that reflects the human
preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize
this estimated reward without drifting too far from the original model.</k

In [25]:
extraction_prompt = """Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all "pieces of knowledge."

# Definition of a "Piece of Knowledge"
A piece of knowledge is statement that:
1. Provides information.
2. Self-Contained
    - It must be fully understandable on its own without needing sentences before or after it. 
    - If it's referring to a term in a previous sentence (e.g., "the reward function"), they should be together as one unit of knowledge.
    - If the sentence is directly building on a previous sentence, either by a pronoun or a reference, they should be together as one unit of knowledge.
2. Atomic and Coherent
    - It should be a single, complete thought.  
    - This can range from a single, dense phrase to a few sentences that are woven together.

# What to Exclude (Do NOT tag these):
Generally speaking, exclude sentences that mostly consist of common words and phrases that don't add much information:
- Paper Structure & Meta-Commentary: Sentences about the paper itself (e.g., "In this section, we describe...", "Figure 3 shows...", "Our contributions are threefold..."). 
- Transitional Sentences: Text that only serves to connect ideas without adding new information (e.g., "Furthermore...", "Now we turn to...", "In addition to the above...").
- Author Speculation or Rhetorical Questions: Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?"). 
- Figures and Tables: Sentences that describe figures and tables.

# Instructions
1.  Read the entire text carefully.
2.  Identify all segments that fit the definition of a "piece of knowledge" and do not fall into the exclusion categories.
3.  Demarcate each piece of knowledge by wrapping it in `<knowledge>` and `</knowledge>` tags.
4.  While you should ensure the knowledge is self-contained and atomic as much as you can, do not just place tags around an entire paragraph. Encapsulate smaller, discrete units of knowledge.
5.  Only return the entire original text with these annotations. Do not modify or summarize the text itself.
"""
# - For instance, it's okay if you have to forgo some portions of a paragraph to break it up into smaller units.
# - Citations: Sentences that primarily exist to cite other work (e.g., "This was previously shown by Smith et al. (2023).").


# Split paper into paragraphs
paper_paragraphs = paper.split('\n\n')

extracted_claims = []
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""{paragraph}"""
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=32768)

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, paragraph) for paragraph in paper_paragraphs]
    extracted_claims = [future.result() for future in futures]

# Print each output
for i, output in enumerate(extracted_claims, 1):
    print_wrapped(f"Paragraph {i}: {output}")
    print_wrapped("-" * 50)

Paragraph 1: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

--------------------------------------------------

Paragraph 2: \begin{abstract} <knowledge>Large-scale unsupervised language models (LMs) learn broad
world knowledge and some reasoning skills.</knowledge> <knowledge>Achieving precise control of the
behavior of large-scale unsupervised LMs is difficult due to the completely unsupervised nature of
their training.</knowledge> <knowledge>Existing methods for gaining steerability in LMs collect
human labels of the relative quality of model generations and fine-tune the unsupervised LM to align
with these preferences, often using reinforcement learning from human feedback (RLHF).</knowledge>
<knowledge>RLHF is a complex and often unstable procedure, which first fits a reward model that
reflects human preferences, and then fine-tunes the large unsupervised LM using reinforcement
learning to maximize this estimated reward without drifting too

In [ ]:
extraction_prompt = r"""Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all "pieces of knowledge."

# Definition of a "Piece of Knowledge"
A piece of knowledge is a sentence that provides information.

# What to Exclude (Do NOT tag these):
- Sentences that are transitional and for structural purposes of the paper, mainly containing language that's generic to any paper, adding zero information e.g. "Our results raise several important questions for future work.", "In this section, we discuss our methodology in relation to other works.
- Transitional Sentences: Text that only serves to connect ideas without adding new information (e.g., "Furthermore...", "Now we turn to...", "In addition to the above...").
- Author Speculation or Rhetorical Questions: Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?"). 
- Figures and Tables: Sentences that describe figures and tables.

# Instructions
1.  Read the entire text carefully.
2.  Identify all sentences that fit the definition of a "piece of knowledge" and do not fall into the exclusion categories.
3.  Please exclude Latex code that is part of tables and figures. You may demarcate any relevant captions of the table or figure.
4.  For sentences that contain latex, place the tags so that it includes any latex code that's part of the sentence e.g. "\begin{definition}" or "\text{...}".
5.  Demarcate each piece of knowledge by wrapping it in `<knowledge>` and `</knowledge>` tags.
6.  Please make sure the tags cover the ENTIRE sentence i.e. the tags are at the beginning and end of the sentence.
7.  Return the entire original text with these annotations. Do not modify or summarize the text itself."""

import re
import pandas as pd

def parse_paper_structure(text):
    """Parse paper into sections, subsections and paragraphs with metadata."""
    sections = []
    
    # Split by sections first
    section_pattern = r'\\section\{([^}]+)\}'
    section_splits = re.split(section_pattern, text)
    
    current_section = "Title/Abstract"
    current_section_content = ""
    
    for i in range(len(section_splits)):
        if i == 0:
            # Content before first section
            content = section_splits[i]
            current_section_content = content
        elif i % 2 == 1:
            # This is a section title
            current_section = section_splits[i]
            continue
        else:
            # This is section content
            content = section_splits[i]
            current_section_content = content
        
        # Now split by subsections within this section
        subsection_pattern = r'\\subsection\{([^}]+)\}'
        subsection_splits = re.split(subsection_pattern, content)
        
        current_subsection = "No Subsection"
        current_subsection_content = ""
        
        for j in range(len(subsection_splits)):
            if j == 0:
                # Content before first subsection
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            elif j % 2 == 1:
                # This is a subsection title
                current_subsection = subsection_splits[j]
                continue
            else:
                # This is subsection content
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            
            # Split into paragraphs
            paragraphs = [p.strip() for p in subsection_content.split('\n\n') if p.strip()]
            
            for paragraph in paragraphs:
                sections.append({
                    'section': current_section,
                    'subsection': current_subsection,
                    'paragraph': paragraph,
                    'section_text': current_section_content,
                    'subsection_text': current_subsection_content
                })
    
    return pd.DataFrame(sections)

# Parse paper structure
paper_df = parse_paper_structure(paper)

# Process each paragraph with LLM
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""{paragraph}"""
    return utils.query_llm(prompt, model='gpt-5')

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, row['paragraph']) for _, row in paper_df.iterrows()]
    extracted_claims = [future.result() for future in futures]

# Add extracted claims to dataframe
paper_df['extracted_claims'] = extracted_claims

# Print each output with section/subsection context
for i, (_, row) in enumerate(paper_df.iterrows(), 1):
    print_wrapped(f"Section: {row['section']}")
    print_wrapped(f"Subsection: {row['subsection']}")
    print_wrapped(f"Paragraph {i}: {row['extracted_claims']}")
    print_wrapped("-" * 50)

### 1. Extracting Knowledge

This is to tag sentences that contain information, knowledge, and facts. From these we will extract self-contained, atomic facts.

In [20]:
extraction_prompt = r"""Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all sentences that contain "pieces of knowledge" or "facts."

# What to Exclude (Do NOT tag these):
- Sentences that are transitional and for structural purposes of the paper, mainly containing language that's generic to any paper, adding zero information e.g. "Our results raise several important questions for future work.", "In this section, we discuss our methodology in relation to other works.
- Author Speculation or Rhetorical Questions: Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?"). 
- Figures and Tables: Latex commands that generate figures and tables.

# Instructions
1.  Read the entire text carefully.
2.  Identify all sentences that contain a "piece of knowledge" or a "fact" and do not fall into the exclusion categories.
3.  While you shouldn't tag tables/figures, you may tag *captions* of the table or figure.
4.  For sentences that contain latex, place the tags so that it includes any latex code that's part of the sentence e.g. "\begin{definition}" or "\text{...}".
5.  Wrap each sentence in `<knowledge>` and `</knowledge>` tags.
6.  Please make sure the tags cover the ENTIRE sentence i.e. the tags are at the beginning and end of the sentence.
7.  Return the entire original text with these annotations. Do not modify or summarize the text itself."""

# Parse paper structure
paper_df = parse_paper_structure(paper)

# Process each paragraph with LLM
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""{paragraph}"""
    return utils.query_llm(prompt, model='gpt-5')

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, row['paragraph']) for _, row in paper_df.iterrows()]
    extracted_claims = [future.result() for future in futures]

# Add extracted claims to dataframe
paper_df['extracted_claims'] = extracted_claims

# Print each output with section/subsection context
for i, (_, row) in enumerate(paper_df.iterrows(), 1):
    print_wrapped(f"Section: {row['section']}")
    print_wrapped(f"Subsection: {row['subsection']}")
    print_wrapped(f"Paragraph {i}: {row['extracted_claims']}")
    print_wrapped("-" * 50)

Section: Title/Abstract

Subsection: No Subsection

Paragraph 1: <knowledge>\title{Direct Preference Optimization: Your Language Model is Secretly a
Reward Model}</knowledge>

--------------------------------------------------

Section: Title/Abstract

Subsection: No Subsection

Paragraph 2: \begin{abstract} <knowledge>While large-scale unsupervised language models (LMs) learn
broad world knowledge and some reasoning skills, achieving precise control of their behavior is
difficult due to the completely unsupervised nature of their training.</knowledge>
<knowledge>Existing methods for gaining such steerability collect human labels of the relative
quality of model generations and fine-tune the unsupervised LM to align with these preferences,
often with reinforcement learning from human feedback (RLHF).</knowledge> <knowledge>However, RLHF
is a complex and often unstable procedure, first fitting a reward model that reflects the human
preferences, and then fine-tuning the large unsupervise

#### 1.1 Check Knowledge is actually in the paper

In [21]:
import re
import pandas as pd

def extract_text_from_knowledge_tags(text: str) -> list[str]:
    """
    Finds all <knowledge> tags in a given text and extracts their content.

    Args:
        text: A string containing the text to parse, which may include
              <knowledge>...</knowledge> tags.

    Returns:
        A list of strings, where each string is the content found within
        a <knowledge> tag. The content is stripped of leading/trailing
        whitespace.
    """
    # This regex pattern finds all content between <knowledge> and </knowledge>.
    # - The (.*?) part is a non-greedy capture group for the content inside the tags.
    # - The re.DOTALL flag allows the '.' character to match newlines, so tags
    #   that span multiple lines are correctly handled.
    pattern = re.compile(r'<knowledge>(.*?)</knowledge>', re.DOTALL)
    
    # re.findall returns a list of all captured groups.
    matches = pattern.findall(text)
    
    # Clean up any leading/trailing whitespace from the extracted text.
    cleaned_matches = [match.strip() for match in matches]
    
    return cleaned_matches

def remove_knowledge_tags(text: str) -> str:
    """Remove knowledge tags from text while preserving the content."""
    pattern = re.compile(r'</?knowledge>', re.DOTALL)
    return pattern.sub('', text)

paper_df['extracted_claims'] = extracted_claims

# Extract a list of knowledge statements for each row
paper_df['knowledge_list'] = paper_df['extracted_claims'].apply(extract_text_from_knowledge_tags)

# Explode the DataFrame on the knowledge_list column
paper_df_exploded = paper_df.explode('knowledge_list').rename(columns={'knowledge_list': 'raw_knowledge_statement'})

# Drop rows with no knowledge statements
paper_df_exploded = paper_df_exploded[paper_df_exploded['raw_knowledge_statement'].notna()]

# Count total claims extracted by the LLM before filtering
total_extracted_claims = paper_df_exploded['raw_knowledge_statement'].notna().sum()

# Filter out claims that are not actually in the original paper text (case-insensitive)
paper_lower = paper.lower()
def is_claim_in_paper(claim):
    # Rows with no knowledge statement (claim is NaN) are kept
    if pd.isna(claim):
        return True
    # Check if the lowercased claim is in the lowercased paper
    return claim.strip().lower() in paper_lower

# Apply the filter and create a new validated dataframe
paper_df_validated = paper_df_exploded[paper_df_exploded['raw_knowledge_statement'].apply(is_claim_in_paper)].copy()

# Count claims that passed validation
validated_claims_count = paper_df_validated['raw_knowledge_statement'].notna().sum()
print(f"Found {validated_claims_count}/{total_extracted_claims} extracted claims in the original paper text.")

# Clean up the original paragraph text by removing knowledge tags from the validated dataframe
paper_df_validated['paragraph'] = paper_df_validated['extracted_claims'].apply(remove_knowledge_tags)

# Drop the now-redundant columns
paper_df_validated = paper_df_validated.drop(columns=['extracted_claims'])

# Display the result
print(f"Total rows after exploding and validation: {len(paper_df_validated)}")
paper_df_validated

Found 212/213 extracted claims in the original paper text.
Total rows after exploding and validation: 212


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement
0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ..."
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...
...,...,...,...,...,...,...
39,Discussion,No Subsection,"Learning from preferences is a powerful, scala...","\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","With virtually no tuning of hyperparameters, D..."
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...",Our initial results suggest that DPO policies ...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6..."
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra..."


In [22]:
# Print all the NAs
na_rows = paper_df_validated[paper_df_validated['raw_knowledge_statement'].isna()]
print(f"Found {len(na_rows)} rows with NA knowledge statements:")
for idx, row in na_rows.iterrows():
    print(f"\nRow {idx}:")
    print(f"Paragraph: {row['paragraph']}")

Found 0 rows with NA knowledge statements:


### [OLD] Filter Bad Probes

This is to avoid figures, tables, and sentences that are filled with citations or dominated by equations. While OLMO has been trained on arxiv documents that include latex, we avoid unlearnable probes that may require further pre-training on mathematical notation and latex. 

In [ ]:
# Filter out knowledge statements that are more than 50% LaTeX
def calculate_latex_percentage(text):
    """
    Calculate the percentage of LaTeX/mathematical content in a text string.
    
    Args:
        text (str): The text to analyze
        
    Returns:
        float: Percentage of text that is LaTeX/mathematical (0-100)
    """
    if pd.isna(text) or not text.strip():
        return 0.0
    
    import re
    
    total_chars = len(text)
    latex_chars = 0
    
    # Count LaTeX commands (backslash followed by letters)
    latex_commands = re.findall(r'\\[a-zA-Z]+', text)
    for cmd in latex_commands:
        latex_chars += len(cmd)
    
    # Remove LaTeX commands to avoid double counting
    text_without_commands = re.sub(r'\\[a-zA-Z]+', '', text)
    
    # Count non-alphabetic characters in the remaining text
    for char in text_without_commands:
        if not char.isalpha() and not char.isspace():
            latex_chars += 1
    
    # Calculate percentage
    latex_percentage = (latex_chars / total_chars) * 100 if total_chars > 0 else 0.0
    
    return latex_percentage

# Apply the filter
paper_df_validated['latex_percentage'] = paper_df_validated['raw_knowledge_statement'].apply(calculate_latex_percentage)

# Filter out statements with more than 50% LaTeX
latex_threshold = 50
paper_df_filtered = paper_df_validated[paper_df_validated['latex_percentage'] <= latex_threshold].copy()

# Report filtering results
total_before = len(paper_df_validated)
total_after = len(paper_df_filtered)
filtered_out = total_before - total_after

print(f"LaTeX filtering results:")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} statements with >{latex_threshold}% LaTeX content")

# Show all filtered statements
if filtered_out > 0:
    high_latex_statements = paper_df_validated[paper_df_validated['latex_percentage'] > latex_threshold]
    print(f"\nAll {filtered_out} filtered statements (high LaTeX content):")
    for idx, row in high_latex_statements.iterrows():
        print(f"  Row {idx}: LaTeX {row['latex_percentage']:.1f}%")
        print(f"    Statement: '{row['raw_knowledge_statement']}'")
        print()
        
paper_df_filtered.reset_index(drop=True, inplace=True)
paper_df_filtered


LaTeX filtering results:
  Before filtering: 203 knowledge statements
  After filtering: 169 knowledge statements
  Filtered out: 34 statements with >20% LaTeX content

All 34 filtered statements (high LaTeX content):
  Row 8: LaTeX 22.2%
    Statement: 'Similarly, \textit{preference-based RL} (PbRL) learns from binary preferences generated by an \textit{unknown} `scoring' function rather than rewards \citep{BusaFekete2014,ruiz2023dueling}.'

  Row 10: LaTeX 29.2%
    Statement: 'We review the RLHF pipeline in \citeauthor{ziegler2020finetuning} (and later \citep{stiennon2022learning, bai2022training, ouyang2022training}).'

  Row 12: LaTeX 22.4%
    Statement: 'In the second phase the SFT model is prompted with prompts $x$ to produce pairs of answers $(y_1, y_2)\sim \pisft(y \mid x)$.'

  Row 12: LaTeX 68.7%
    Statement: '\begin{equation}\label{eq:bradley-terry}
    p^*(y_1\succ y_2 \mid x)=\frac{\exp\left(r^*(x, y_1)\right)}{\exp\left(r^*(x, y_1)\right) + \exp\left(r^*(x, y_2)\right

,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,latex_percentage
0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,10.465116
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,2.222222
2,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,2.074689
3,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...",1.773050
4,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,0.833333
...,...,...,...,...,...,...,...
164,Discussion,No Subsection,"Learning from preferences is a powerful, scala...","\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","With virtually no tuning of hyperparameters, D...",1.687764
165,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...",Our initial results suggest that DPO policies ...,2.290076
166,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...",3.846154
167,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...",3.208556


In [8]:
print(paper_df_filtered.loc[37,'raw_knowledge_statement'])

Contextual bandit learning using preferences or rankings of actions, rather than rewards, is known as a contextual dueling bandit (CDB; \cite{yue2012karmed,dudik2015contextual}).


Remove Citations


In [89]:


def remove_trailing_citation(text: str) -> str:
    """
    Removes a trailing LaTeX citation (e.g., \cite{...} or \citep{...}) 
    from a string if it exists.
    """
    if not isinstance(text, str):
        return text
    
    # Regex to find \cite{...} or \citep{...} at the end of a string,
    # allowing for whitespace before it and an optional period after it.
    pattern = r'\s*~?\\(cite|citep)\{[^}]*\}\.?$'
    
    return re.sub(pattern, '.', text).strip()

# --- Example Usage ---

# 1. Create a sample DataFrame. 
#    (You would use your own DataFrame here)
data = {
    'raw_knowledge_statement': [
        'This is a statement with a citation \\cite{author2023}',
        'This is another statement with a citation and period \\citep{anotherauthor2022}.',
        'This statement has no citation.',
        'This statement has a citation \\cite{somepaper} in the middle.',
        'This one is clean.',
        'Another one ending with a citation \\cite{end}',
        None, # Example of a non-string value
        42    # Example of a non-string value
    ]
}

temp_df = pd.DataFrame(data)

# Test it
temp_df['statement_cleaned'] = temp_df['raw_knowledge_statement'].apply(remove_trailing_citation)
print(temp_df[['raw_knowledge_statement', 'statement_cleaned']])

def remove_trailing_parentheses(text: str) -> str:
    """
    Removes content in parentheses at the end of a string if it exists.
    e.g., "This is a sentence (with some text)." -> "This is a sentence"
    """
    if not isinstance(text, str):
        return text
        
    # Regex to find parentheses at the end of a string,
    # allowing for whitespace before it and an optional period after it.
    pattern = r'\s*\([^)]*\)\.?$'

    return re.sub(pattern, '', text).strip()

# Apply it to our df
paper_df_filtered['raw_knowledge_statement'] = paper_df_filtered['raw_knowledge_statement'].apply(remove_trailing_citation).apply(remove_trailing_parentheses)


                             raw_knowledge_statement  \
0  This is a statement with a citation \cite{auth...   
1  This is another statement with a citation and ...   
2                    This statement has no citation.   
3  This statement has a citation \cite{somepaper}...   
4                                 This one is clean.   
5      Another one ending with a citation \cite{end}   
6                                               None   
7                                                 42   

                                   statement_cleaned  
0               This is a statement with a citation.  
1  This is another statement with a citation and ...  
2                    This statement has no citation.  
3  This statement has a citation \cite{somepaper}...  
4                                 This one is clean.  
5                Another one ending with a citation.  
6                                               None  
7                                                 42  


In [90]:
print(paper_df_filtered.loc[37,'raw_knowledge_statement'])

Contextual bandit learning using preferences or rankings of actions, rather than rewards, is known as a contextual dueling bandit


Filter out probes with unsuitable targets

In [ ]:
from tqdm import tqdm
from importlib import reload
reload(utils)
import json
# Evaluate knowledge statements for suitability
prompt = {}
prompt['system'] = """You are an expert at evaluating knowledge statements for use as memory probes in LLM training. Your task is to determine whether a knowledge statement from an academic paper is suitable for testing an LLM's factual recall.

A knowledge statement is UNSUITABLE if it:
1. Ends with continuous LaTeX code or mathematical notation that would prevent using the end as a fill-in-the-blank target. If it only contains a few characters of LaTeX, it is still suitable.
2. The knowledge statement is only an internal pointer to a part of the paper (e.g., "This is discussed in Section 3.1").
3. The knowledge statement is a rhetorical question.
4. The knowledge statement is cutoff in the middle or shows any displays of corruption.

Respond with JSON format with the following keys:
- "suitable": boolean (true/false)
- "unsuitable_condition": integer (1, 2, or 3) or null if suitable"""

# Apply evaluation to each knowledge statement
print("Evaluating knowledge statement suitability...")
suitability_results = []
unsuitable_conditions = []

for idx, row in tqdm(paper_df_filtered.iterrows(), total=len(paper_df_filtered)):
    statement = row['raw_knowledge_statement']
    user_prompt = f"Preceding Context: {row['paragraph'].split(statement)[0]}\n\nKnowledge statement: {statement}"
    if idx == 0:
        print(user_prompt)
    full_prompt = {
        'system': prompt['system'],
        'user': user_prompt
    }
    
    result = utils.query_llm(full_prompt, model='gpt-4.1-mini', return_json=True, max_tokens=50)
    result = json.loads(result)
    is_suitable = result['suitable']
    unsuitable_condition = result.get('unsuitable_condition')
    
    suitability_results.append(is_suitable)
    unsuitable_conditions.append(unsuitable_condition)

# Add suitability column and filter
paper_df_filtered['is_suitable'] = suitability_results
paper_df_filtered['unsuitable_condition'] = unsuitable_conditions
paper_df_suitable = paper_df_filtered[paper_df_filtered['is_suitable']].copy()

# Report filtering results
total_before = len(paper_df_filtered)
total_after = len(paper_df_suitable)
filtered_out = total_before - total_after

print(f"\nSuitability filtering results:")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} unsuitable statements")


In [92]:
# Show all unsuitable statements with their conditions
if filtered_out > 0:
    unsuitable_statements = paper_df_filtered[~paper_df_filtered['is_suitable']]
    print(f"\nAll {filtered_out} unsuitable statements:")
    for idx, row in unsuitable_statements.iterrows():
        condition = row['unsuitable_condition']
        print_wrapped(f"  Row {idx} (Condition {condition}): '{row['raw_knowledge_statement']}'")


All 35 unsuitable statements:
  Row 45 (Condition 1.0): '\textbf{Reward Modelling Phase}: In the second phase the SFT model is
prompted with prompts $x$ to produce pairs of answers $(y_1, y_2)\sim \pisft(y \mid x)$.'

  Row 49 (Condition 1.0): 'The BT model stipulates that the human preference distribution $p^*$ can
be written as:'

  Row 51 (Condition 1.0): 'Framing the problem as a binary classification we have the negative log-
likelihood loss:'

  Row 52 (Condition 1.0): 'where $\sigma$ is the logistic function.'

  Row 65 (Condition 2.0): 'We start with the same RL objective as prior work, Eq.~\ref{eq:RL}, under
a general reward function $r$.'

  Row 66 (Condition 2.0): 'Following prior work~\citep{peters2007reinforcement, peng2019advantage,
korbak2022reinforcement, go2023aligning}, it is straightforward to show that the optimal solution to
the KL-constrained reward maximization objective in Eq.~\ref{eq:RL} takes the form:'

  Row 69 (Condition 2.0): 'Specifically, we first take 

### 2. Extract Self-Contained, Atomic Facts
Given the original, source sentences from the paper, we break each sentence into the parts that presents a new fact.


In [ ]:
import concurrent.futures
# You will be given a piece of text. The text is written such that each sentence is contextualized, which is not desirable for breaking the text apart into stand alone facts. Your task is to extract and rewrite the information into self-contained, atomic facts and that can be separated into a probe and a target. 

# First, segment the text into coherent subtexts. For each subtext, extract at most three *self-contained, atomic facts*, each a single declarative claim that can stand alone. Rewrite every fact so that it ends with a 1–3 word *target* capturing the key information. The *probe* is the same sentence with the *target* removed; it should read naturally and implicitly ask for the missing key information. 

# - Prefer precise, contentful targets (e.g., “reasoning path”, “human feedback”) over generic terms; keep targets to 1–3 words only. 
# - Do not over-extract: include only facts warranted by the text and extract at most 3 facts per subtext.
# - Preserve the original meaning and scope, avoid introducing new claims, and ensure the last words of each sentence are exactly the target. 
# - Present your results in the demonstrated Probe/Target format.
#I am trying to create self-contained, atomic knowledge probes for a language model, to measure its ability to recall facts. 
def extract_atomic_facts(sentence, paragraph):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For more complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. Often times, there are multiple valid targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

# Demonstrations
The next few demonstrations will be based on the same paragraph. 

Paragraph: "Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning in language models.
\begin{enumerate}[topsep=1pt,itemsep=0ex]%
    \item First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps.
    \item Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question).
    \item Third, chain-of-thought reasoning can be used for tasks such as math word problems, commonsense reasoning, and symbolic manipulation, and is potentially applicable (at least in principle) to any task that humans can solve via language.
    \item Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting.
\end{enumerate}"

### Example 1
Sentence: "First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps." 

Targets:
- "intermediate steps"
- "reasoning steps"

Output:
- Probe: "Chain of thought allows models to decompose multi-step problems into", Target: "intermediate steps"
- Probe: "By decomposing multi-step problems into intermediate steps, chain of thought allows additional computation to be allocated to problems that require more", Target: "reasoning steps"

### Example 2
Sentence: "Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question)."

Targets:
- "interpretable window"
- "reasoning path"

Output:
- Probe: "Chain-of-thought shows how the model formed an answer and provides opportunities to debug mistakes in the", Target: "reasoning path"
- Probe: "The behavior of language models can be difficult to characterize but chain-of-thought provides an", Target: "interpretable window"

### Example 3
Sentence: "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting."

Targets:
- "including examples"
- "elicited"

Output:
- Probe: "Chain-of-thought reasoning can be elicited in sufficiently large off-the-shelf language models simply by", Target: "including examples"
- Probe: "By including examples of chain of thought sequences, chain-of-thought reasoning in sufficiently large off-the-shelf language models can be readily", Target: "elicited"

The next few examples will be based on the following paragraph for context. 
Paragraph: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

### Example 4
Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

Targets:
- "world knowledge"
- "reasoning skills"
- "precise control"
- "behavior"
- "unsupervised training"

Output:
- Probe: "Large-scale unsupervised LMs learn broad", Target: "world knowledge" 
- Probe: "Large-scale unsupervised LMs learn", Target: "reasoning skills" 
- Probe: NA for "precise control" (Difficult to place at end of sentence)
- Probe: "Due to the completely unsupervised nature of the training of large-scale unsupervised language models (LMs), it is difficult to achieve precise control of their", Target: "behavior"
- Probe: "Achieving precise control of the behavior of unsupervised language models is difficult due to their", Target: "unsupervised training"

### Example 5
Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Targets:
- "human labels"
- "model generations"
- "preferences"

Output:
- Probe: "To steer unsupervised language models, existing methods collect", Target: "human labels"
- Probe: "Existing methods for gaining such steerability collect human labels of the relative quality of", Target: "model generations"
- Probe: "Existing methods align unsupervised language models by fine-tuning on human", Target: "preferences"
"""
    prompt['user'] = f"""### Input\nContext: {paragraph}\n\nSentence: {sentence}### Output\n"""
    if len(paragraph.strip()) < 50:
        return None
    return utils.query_llm(prompt, model='gpt-5')

# Process first row only for testing
first_row = paper_df_validated.iloc[2]
atomic_facts = extract_atomic_facts(first_row['raw_knowledge_statement'], first_row['paragraph'])

print(f"Extracted atomic facts for first row: {atomic_facts}")

Extracted atomic facts for first row: - Probe: "To steer unsupervised language models, existing methods collect", Target: "human labels"
- Probe: "Existing methods for increasing steerability collect human labels of the relative quality of", Target: "model generations"
- Probe: "To align the behavior of unsupervised language models with desired outcomes, existing methods fine-tune on human", Target: "preferences"
- Probe: "Fine-tuning to align unsupervised language models with human preferences is often performed using reinforcement learning from human feedback, abbreviated as", Target: "RLHF"


In [ ]:
import concurrent.futures
# You will be given a piece of text. The text is written such that each sentence is contextualized, which is not desirable for breaking the text apart into stand alone facts. Your task is to extract and rewrite the information into self-contained, atomic facts and that can be separated into a probe and a target. 

# First, segment the text into coherent subtexts. For each subtext, extract at most three *self-contained, atomic facts*, each a single declarative claim that can stand alone. Rewrite every fact so that it ends with a 1–3 word *target* capturing the key information. The *probe* is the same sentence with the *target* removed; it should read naturally and implicitly ask for the missing key information. 

# - Prefer precise, contentful targets (e.g., “reasoning path”, “human feedback”) over generic terms; keep targets to 1–3 words only. 
# - Do not over-extract: include only facts warranted by the text and extract at most 3 facts per subtext.
# - Preserve the original meaning and scope, avoid introducing new claims, and ensure the last words of each sentence are exactly the target. 
# - Present your results in the demonstrated Probe/Target format.
#I am trying to create self-contained, atomic knowledge probes for a language model, to measure its ability to recall facts. 
def extract_atomic_facts(sentence, paragraph):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For more complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. Often times, there are multiple valid targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

# Demonstrations
The next few demonstrations will be based on the same paragraph. 

Paragraph: "Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning in language models.
\begin{enumerate}[topsep=1pt,itemsep=0ex]%
    \item First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps.
    \item Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question).
    \item Third, chain-of-thought reasoning can be used for tasks such as math word problems, commonsense reasoning, and symbolic manipulation, and is potentially applicable (at least in principle) to any task that humans can solve via language.
    \item Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting.
\end{enumerate}"

### Example 1
Sentence: "First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps." 

Targets:
- "intermediate steps"
- "reasoning steps"

Contextualized Probes:
- Probe: "Chain of thought allows models to decompose multi-step problems into", Target: "intermediate steps"
- Probe: "By decomposing multi-step problems into intermediate steps, chain of thought allows additional computation to be allocated to problems that require more", Target: "reasoning steps"

### Example 2
Sentence: "Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question)."

Targets:
- "interpretable window"
- "reasoning path"

Contextualized Probes:
- Probe: "Chain-of-thought shows how the model formed an answer and provides opportunities to debug mistakes in the", Target: "reasoning path"
- Probe: "The behavior of language models can be difficult to characterize but chain-of-thought provides an", Target: "interpretable window"

### Example 3
Sentence: "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting."

Targets:
- "including examples"
- "elicited"

Contextualized Probes:
- Probe: "Chain-of-thought reasoning can be elicited in sufficiently large off-the-shelf language models simply by", Target: "including examples"
- Probe: "By including examples of chain of thought sequences, chain-of-thought reasoning in sufficiently large off-the-shelf language models can be readily", Target: "elicited"

The next few examples will be based on the following paragraph for context. 
Paragraph: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

### Example 4
Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

Targets:
- "world knowledge"
- "reasoning skills"
- "precise control"
- "behavior"
- "unsupervised training"

Contextualized Probes:
- Probe: "Large-scale unsupervised LMs learn broad", Target: "world knowledge" 
- Probe: "Large-scale unsupervised LMs learn", Target: "reasoning skills" 
- Probe: NA (Difficult to place "precise control" at end of sentence)
- Probe: "Due to the completely unsupervised nature of the training of large-scale unsupervised language models (LMs), it is difficult to achieve precise control of their", Target: "behavior"
- Probe: "Achieving precise control of the behavior of unsupervised language models is difficult due to their", Target: "unsupervised training"

### Example 5
Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Targets:
- "human labels"
- "model generations"
- "preferences"

Contextualized Probes:
- Probe: "To steer unsupervised language models, existing methods collect", Target: "human labels"
- Probe: "Existing methods for gaining such steerability collect human labels of the relative quality of", Target: "model generations"
- Probe: "Existing methods align unsupervised language models by fine-tuning on human", Target: "preferences"
"""
    prompt['user'] = f"""### Input\nParagraph: {paragraph}\n\nSentence: {sentence}\n\n"""
    if len(paragraph.strip()) < 50:
        return None
    return utils.query_llm(prompt, model='gpt-5')

# Process first row only for testing
first_row = paper_df_validated.iloc[2]
atomic_facts = extract_atomic_facts(first_row['raw_knowledge_statement'], first_row['paragraph'])

print(f"Extracted atomic facts for first row: {atomic_facts}")

Extracted atomic facts for first row: - Probe: To steer unsupervised language models, existing methods collect, Target: human labels
- Probe: Existing methods for steering unsupervised language models collect human labels of the relative quality of, Target: model generations
- Probe: Existing methods align unsupervised language models by fine-tuning them to match human, Target: preferences
- Probe: To align unsupervised language models with human preferences, many approaches use reinforcement learning from human feedback, abbreviated as, Target: RLHF
- Probe: For outputs produced by language models, researchers collect labels from humans that assess the, Target: relative quality


In [ ]:
import concurrent.futures
# You will be given a piece of text. The text is written such that each sentence is contextualized, which is not desirable for breaking the text apart into stand alone facts. Your task is to extract and rewrite the information into self-contained, atomic facts and that can be separated into a probe and a target. 

# First, segment the text into coherent subtexts. For each subtext, extract at most three *self-contained, atomic facts*, each a single declarative claim that can stand alone. Rewrite every fact so that it ends with a 1–3 word *target* capturing the key information. The *probe* is the same sentence with the *target* removed; it should read naturally and implicitly ask for the missing key information. 

# - Prefer precise, contentful targets (e.g., “reasoning path”, “human feedback”) over generic terms; keep targets to 1–3 words only. 
# - Do not over-extract: include only facts warranted by the text and extract at most 3 facts per subtext.
# - Preserve the original meaning and scope, avoid introducing new claims, and ensure the last words of each sentence are exactly the target. 
# - Present your results in the demonstrated Probe/Target format.
#I am trying to create self-contained, atomic knowledge probes for a language model, to measure its ability to recall facts. 
def extract_atomic_facts(sentence, paragraph):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For more complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. Often times, there are multiple valid targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

# Demonstrations
The next few demonstrations will be based on the same paragraph. 

Paragraph: "Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning in language models.
\begin{enumerate}[topsep=1pt,itemsep=0ex]%
    \item First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps.
    \item Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question).
    \item Third, chain-of-thought reasoning can be used for tasks such as math word problems, commonsense reasoning, and symbolic manipulation, and is potentially applicable (at least in principle) to any task that humans can solve via language.
    \item Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting.
\end{enumerate}"

### Example 1
Sentence: "First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps." 

Targets:
- "intermediate steps"
- "reasoning steps"

Contextualized Probes:
- Probe: "Chain of thought allows models to decompose multi-step problems into", Target: "intermediate steps"
- Probe: "By decomposing multi-step problems into intermediate steps, chain of thought allows additional computation to be allocated to problems that require more", Target: "reasoning steps"

### Example 2
Sentence: "Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question)."

Targets:
- "interpretable window"
- "reasoning path"

Contextualized Probes:
- Probe: "Chain-of-thought shows how the model formed an answer and provides opportunities to debug mistakes in the", Target: "reasoning path"
- Probe: "The behavior of language models can be difficult to characterize but chain-of-thought provides an", Target: "interpretable window"

### Example 3
Sentence: "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting."

Targets:
- "including examples"
- "elicited"

Contextualized Probes:
- Probe: "Chain-of-thought reasoning can be elicited in sufficiently large off-the-shelf language models simply by", Target: "including examples"
- Probe: "By including examples of chain of thought sequences, chain-of-thought reasoning in sufficiently large off-the-shelf language models can be readily", Target: "elicited"

The next few examples will be based on the following paragraph for context. 
Paragraph: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

### Example 4
Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

Targets:
- "world knowledge"
- "reasoning skills"
- "precise control"
- "behavior"
- "unsupervised training"

Contextualized Probes:
- Probe: "Large-scale unsupervised LMs learn broad", Target: "world knowledge" 
- Probe: "Large-scale unsupervised LMs learn", Target: "reasoning skills" 
- Probe: NA (Difficult to place "precise control" at end of sentence)
- Probe: "Due to the completely unsupervised nature of the training of large-scale unsupervised language models (LMs), it is difficult to achieve precise control of their", Target: "behavior"
- Probe: "Achieving precise control of the behavior of unsupervised language models is difficult due to their", Target: "unsupervised training"

### Example 5
Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Targets:
- "human labels"
- "model generations"
- "preferences"

Contextualized Probes:
- Probe: "To steer unsupervised language models, existing methods collect", Target: "human labels"
- Probe: "Existing methods for gaining such steerability collect human labels of the relative quality of", Target: "model generations"
- Probe: "Existing methods align unsupervised language models by fine-tuning on human", Target: "preferences"
"""
    prompt['user'] = f"""### Input\nParagraph: {paragraph}\n\nSentence: {sentence}\n\n"""
    if len(paragraph.strip()) < 50:
        return None
    return utils.query_llm(prompt, model='gpt-5')

# Process first row only for testing
first_row = paper_df_validated.iloc[2]
atomic_facts = extract_atomic_facts(first_row['raw_knowledge_statement'], first_row['paragraph'])

print(f"Extracted atomic facts for first row: {atomic_facts}")

In [34]:
paper_df_validated.iloc[15]['raw_knowledge_statement']

'At a high level, existing methods instill the desired behaviors into a language model using curated sets of human preferences representing the types of behaviors that humans find safe and helpful.'

In [31]:
# Process first row only for testing
first_row = paper_df_validated.iloc[15]
atomic_facts = extract_atomic_facts(first_row['raw_knowledge_statement'], first_row['paragraph'])

print(f"Extracted atomic facts for first row: {atomic_facts}")

Extracted atomic facts for first row: - Probe: "To elicit safe and helpful behavior, existing methods for training language models use curated sets of", Target: "human preferences"
- Probe: "In preference learning for language models, curated sets of human preferences are used to instill the", Target: "desired behaviors"
- Probe: "Curated sets of human preferences representing behaviors that humans find safe and helpful are used to instill desired behaviors into", Target: "language models"
- Probe: "In language model alignment methods, safe and helpful behaviors defined by curated human preference datasets are", Target: "instilled"
- Probe: "To align model behavior with what humans consider safe and helpful, existing methods rely on curated human", Target: "preferences"


You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often found in the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. There are often multiple targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

----

prompt['system'] = """You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the key information of a sentence i.e. the target. For instance, in the sentence, "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Apply this same reductionary thought process that I've demonstrated here to identify the targets. Often times, there are multiple valid targets, but only select the targets such that the sentence can be paragraphased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that the information is true for. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

----


Write each fact as one declarative sentence whose final 1–3 words form the target capturing the key information; the probe is the same sentence with the final target span removed and should read naturally as a cloze. Prefer precise, contentful targets; ensure the last words are exactly the target; avoid cross-references to other facts; preserve the original meaning; extract at most three facts per sentence; and present results in the demonstrated Probe/Target format.

    prompt['system'] = """You will be given two inputs: (1) a full paragraph for context and (2) a single sentence drawn from that paragraph. Your task is to rewrite that sentence into 1–3 self-contained, atomic facts that can stand entirely on their own as probe–target pairs. Use the paragraph only to supply whatever context is needed to make each fact standalone; explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers. Also, do not introduce information not entailed by the sentence. Write each fact as one declarative sentence whose final 1–3 words form the target capturing the key information; the probe is the same sentence with the final target span removed and should read naturally as a cloze. Prefer precise, contentful targets; ensure the last words are exactly the target; avoid cross-references to other facts; preserve the original meaning; extract at most three facts per sentence; and present results in the demonstrated Probe/Target format.

-----
"You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often found in the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. There are often multiple targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target."

Adjust this prompt above to request the following: "take the output and double check if the probes have properly satisfied the conditions or can be even improved i.e. can be formatted in a more natural way, stays true to the original meaning of the sentence, fully contextualizes the sentence to be self-contained, the target is non-trivial and a central part of the sentence, and is rewritten to naturally place the target at the end. Please filter out probes that that don't meet these conditions and refine the remaining probes."


### 3. Filter and Refine probes

In [ ]:
import concurrent.futures
 
def validate_atomic_facts(sentence, paragraph):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

# Demonstrations
The next few demonstrations will be based on the same paragraph. 

Paragraph: "Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning in language models.
\begin{enumerate}[topsep=1pt,itemsep=0ex]%
    \item First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps.
    \item Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question).
    \item Third, chain-of-thought reasoning can be used for tasks such as math word problems, commonsense reasoning, and symbolic manipulation, and is potentially applicable (at least in principle) to any task that humans can solve via language.
    \item Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting.
\end{enumerate}"

### Example 1
Sentence: "First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps." 

Targets:
- "intermediate steps"
- "reasoning steps"

Contextualized Probes:
- Probe: "Chain of thought allows models to decompose multi-step problems into", Target: "intermediate steps"
- Probe: "By decomposing multi-step problems into intermediate steps, chain of thought allows additional computation to be allocated to problems that require more", Target: "reasoning steps"

### Example 2
Sentence: "Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question)."

Targets:
- "interpretable window"
- "reasoning path"

Contextualized Probes:
- Probe: "Chain-of-thought shows how the model formed an answer and provides opportunities to debug mistakes in the", Target: "reasoning path"
- Probe: "The behavior of language models can be difficult to characterize but chain-of-thought provides an", Target: "interpretable window"

### Example 3
Sentence: "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting."

Targets:
- "including examples"
- "elicited"

Contextualized Probes:
- Probe: "Chain-of-thought reasoning can be elicited in sufficiently large off-the-shelf language models simply by", Target: "including examples"
- Probe: "By including examples of chain of thought sequences, chain-of-thought reasoning in sufficiently large off-the-shelf language models can be readily", Target: "elicited"

The next few examples will be based on the following paragraph for context. 
Paragraph: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

### Example 4
Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

Targets:
- "world knowledge"
- "reasoning skills"
- "precise control"
- "behavior"
- "unsupervised training"

Contextualized Probes:
- Probe: "Large-scale unsupervised LMs learn broad", Target: "world knowledge" 
- Probe: "Large-scale unsupervised LMs learn", Target: "reasoning skills" 
- Probe: NA (Difficult to place "precise control" at end of sentence)
- Probe: "Due to the completely unsupervised nature of the training of large-scale unsupervised language models (LMs), it is difficult to achieve precise control of their", Target: "behavior"
- Probe: "Achieving precise control of the behavior of unsupervised language models is difficult due to their", Target: "unsupervised training"

### Example 5
Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Targets:
- "human labels"
- "model generations"
- "preferences"

Contextualized Probes:
- Probe: "To steer unsupervised language models, existing methods collect", Target: "human labels"
- Probe: "Existing methods for gaining such steerability collect human labels of the relative quality of", Target: "model generations"
- Probe: "Existing methods align unsupervised language models by fine-tuning on human", Target: "preferences"
"""
    prompt['user'] = f"""### Input\nParagraph: {paragraph}\n\nSentence: {sentence}\n\n"""
    if len(paragraph.strip()) < 50:
        return None
    return utils.query_llm(prompt, model='gpt-5')

# Process first row only for testing
first_row = paper_df_validated.iloc[2]
atomic_facts = validate_atomic_facts(first_row['raw_atomic_probes'], first_row['paragraph'])

print(f"Extracted atomic facts for first row: {atomic_facts}")

### Paraphrase Raw Knowledge Statement

In [5]:
import pandas as pd
from tqdm import tqdm
import asyncio
import aiohttp
from concurrent.futures import ThreadPoolExecutor
import threading


def paraphrase_knowledge_statement(statement, paragraph_context, llm_querier):
    """
    Generates a paraphrased version of a knowledge statement using an LLM.

    Args:
        statement (str): The knowledge statement to paraphrase.
        paragraph_context (str): The paragraph context for the statement.
        llm_querier (function): A function that takes a prompt and returns the LLM's response.

    Returns:
        str: The paraphrased knowledge statement.
    """
    prompt = {
        "system": (
            "You are an expert at paraphrasing sentences. Your task is to paraphrase the given knowledge statement "
            "while preserving its meaning and ensuring it flows linguistically within the given paragraph context. "
            "The paraphrased statement should be a single, complete sentence that maintains the same factual content. "
            "Return only the paraphrased sentence, no other text."
        ),
        "user": (
            f"Paragraph context: '{paragraph_context}'\n\n"
            f"Knowledge statement to paraphrase: '{statement}'\n\n"
            f"Paraphrased statement:"
        )
    }
    
    paraphrased_statement = llm_querier(prompt)
    
    # Clean up the response
    return paraphrased_statement.strip()

def generate_paraphrases_for_statement(row, num_paraphrases=5):
    """Generate multiple unique paraphrases for a single knowledge statement."""
    statement = row['raw_knowledge_statement']
    paragraph = row['paragraph']
    
    paraphrases = set()
    max_rounds = 4 # Maximum number of rounds to try
    
    for round_num in range(max_rounds):
        if len(paraphrases) >= num_paraphrases:
            break
            
        needed = num_paraphrases - len(paraphrases)
        
        with ThreadPoolExecutor(max_workers=num_paraphrases) as executor:
            futures = [
                executor.submit(
                    paraphrase_knowledge_statement, 
                    statement, 
                    paragraph,
                    lambda prompt: utils.query_llm(prompt, model="gpt-4.1-mini", temperature=1.75, top_p=0.985)
                )
                for _ in range(needed)
            ]
            
            for future in futures:
                result = future.result()
                if result and result != statement:  # Ensure it's different from original
                    paraphrases.add(result)
        if len(paraphrases) < num_paraphrases:
            print(f"In round {round_num}, {len(paraphrases)} paraphrases generated for row {row.name}... trying again")
    
    if len(paraphrases) < num_paraphrases:
        print(f"WARNING: Only {len(paraphrases)} paraphrases generated for row {row.name}")
    return list(paraphrases)[:num_paraphrases]

# --- Main Execution ---

# Initialize tqdm for progress tracking with pandas
tqdm.pandas(desc="Paraphrasing knowledge statements")

# Apply the paraphrasing function to each row of the DataFrame
paper_df_validated['paraphrased_knowledge_statements'] = paper_df_validated.progress_apply(
    generate_paraphrases_for_statement,
    axis=1
)

# Display the DataFrame with the new column
print(paper_df_validated[['raw_knowledge_statement', 'paragraph', 'paraphrased_knowledge_statements']].head())

Paraphrasing knowledge statements:   0%|          | 0/204 [00:00<?, ?it/s]

In round 0, 4 paraphrases generated for row 0... trying again


Paraphrasing knowledge statements:   1%|          | 2/204 [00:02<03:40,  1.09s/it]

In round 0, 4 paraphrases generated for row 1... trying again


Paraphrasing knowledge statements:   1%|          | 2/204 [00:04<07:26,  2.21s/it]


KeyboardInterrupt: 

In [11]:
# Drop rows with less than 4 paraphrased statements
print("Rows being dropped (less than 4 paraphrases):")
rows_to_drop = paper_df_validated[paper_df_validated['paraphrased_knowledge_statements'].apply(len) < 5]
for idx, row in rows_to_drop.iterrows():
    num_paraphrases = len(row['paraphrased_knowledge_statements'])
    print(f"Row {idx}: {num_paraphrases} paraphrases - {row['raw_knowledge_statement'][:100]}...")

# Drop the rows
paper_df_validated = paper_df_validated[paper_df_validated['paraphrased_knowledge_statements'].apply(len) >= 4]

print(f"\nDropped {len(rows_to_drop)} rows. Remaining: {len(paper_df_validated)} rows")

# Show comparison for first 10 rows
print("\nComparison of original vs paraphrased statements (first 10 rows):")
for idx, row in paper_df_validated.head(10).iterrows():
    print(f"\nRow {idx}:")
    print(f"Original: {row['raw_knowledge_statement']}")
    print("Paraphrases:")
    for i, paraphrase in enumerate(row['paraphrased_knowledge_statements']):
        print(f"  {i+1}. {paraphrase}")


Rows being dropped (less than 4 paraphrases):
Row 22: 4 paraphrases - We begin with by defining an equivalence relation between reward functions....
Row 30: 3 paraphrases - Our experiments use two different approaches to evaluation....
Row 33: 3 paraphrases - This result is particularly notable for multiple reasons....
Row 34: 4 paraphrases - DPO, PPO and Preferred-FT all fine-tune the same GPT-J SFT model\footnote{\url{https://huggingface.c...
Row 37: 4 paraphrases - The results are presented in Table~\ref{tab:ood}....

Dropped 5 rows. Remaining: 201 rows

Comparison of original vs paraphrased statements (first 10 rows):

Row 0:
Original: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}
Paraphrases:
  1. \title{Direct Preference Optimization: Revealing Your Language Model as an Implicit Reward Model}
  2. \title{Direct Preference Optimization: Revealing that Your Language Model Functions Intrinsically as a Reward Model}
  3. \title{Direct Preferen

### [NOT USED] Now extract all unnecessary latex code from these knowledge statements and call the column "knowledge_statement"

In [94]:

# --- NEW: Clean unnecessary LaTeX from knowledge statements ---
cleaning_prompt = r"""Your task is to remove unmeaningful LaTeX commands while keeping the sentence same otherwise.

# Rules:
1.  **Keep Math**: Preserve mathematical notation written in LaTeX (e.g., `$\mathcal{L}$`, `\pi_\theta`, `\mathbb{E}`). Do not expand or remove them.
2.  **Remove Formatting**: Remove formatting commands like `\textbf{...}`, `\textit{...}`, `\texttt{...}`, etc., but keep the text inside them.
3.  **Handle Citations & References**: Simplify citation commands (e.g., `\cite{...}`, `\citep{...}`) and reference commands (e.g., `\ref{...}`) by removing them but keeping the sentence flow. For example, 'See Appendix~\ref{app:1}' becomes 'See Appendix'.
4.  **Keep Core Content**: The final output must be the original sentence otherwise.
5.  Return only the cleaned sentence, with no extra explanations.

# Example 1:
Input: `Our experiments show that \textbf{DPO} can fine-tune LMs to align with human preferences as well as or better than existing methods.`
Output: `Our experiments show that DPO can fine-tune LMs to align with human preferences as well as or better than existing methods.`

# Example 2:
Input: `The gradient with respect to the parameters $\theta$ can be written as:`
Output: `The gradient with respect to the parameters $\theta$ can be written as:`

# Example 3:
Input: `See Appendix~\ref{app:derivation1} for a complete derivation.`
Output: `See Appendix for a complete derivation.`

# Task:
Now, process the following sentence.
"""

def query_single_clean(statement):
    if statement is None:
        return None
    """Query the LLM to clean a single knowledge statement."""
    prompt = {
        'system': cleaning_prompt,
        'user': f"Input: `{statement}`"
    }
    # Assuming 'gpt-4.1' is a valid model alias in your utils
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=2048)

paper_df_validated.dropna(inplace=True)
# Get unique non-null statements to avoid redundant API calls
statements_to_clean = paper_df_validated['raw_knowledge_statement'].unique().tolist()
cleaned_statements = []

print(f"Cleaning {len(statements_to_clean)} unique knowledge statements...")
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(query_single_clean, stmt) for stmt in statements_to_clean]
    cleaned_statements = [future.result() for future in futures]

# Create a mapping from raw statement to cleaned statement
cleaning_map = dict(zip(statements_to_clean, cleaned_statements))

# Map the cleaned statements back to the dataframe
paper_df_validated['knowledge_statement'] = paper_df_validated['raw_knowledge_statement'].map(cleaning_map)

paper_df_validated.reset_index(drop=True, inplace=True)
paper_df_validated

Cleaning 209 unique knowledge statements...


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,knowledge_statement
0,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining such steerability...
2,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...","However, RLHF is a complex and often unstable ..."
3,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,In this paper we introduce a new parameterizat...
4,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"The resulting algorithm, which we call \textit...","The resulting algorithm, which we call Direct ..."
...,...,...,...,...,...,...,...
204,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...","For example, can training with self-labeling f..."
205,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...","On another front, how does reward over-optimiz..."
206,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...","Additionally, while we evaluate models up to 6..."
207,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...","Regarding evaluations, we find that the win ra..."


### [NOT USED] Identify Context Required for each Probe

In [50]:
# --- NEW: Add context to knowledge statements ---

context_prompt = r"""Your task is to extract a concise, self-contained piece of information from a larger text. You will be given a `subsection_text` and a `knowledge_statement` that is a direct quote from the text.

Your goal is to expand the `knowledge_statement` by including the minimal necessary preceding context from the `subsection_text` to make it understandable on its own. The output should be a single, contiguous block of text copied "word-for-word" from the `subsection_text`.

# Instructions:
1.  The `knowledge_statement` must be at the end of your output.
2.  The output must be a direct, continuous excerpt from the `subsection_text`. Do not add, remove, or change any words.
3.  Include only the *minimal* amount of preceding text required for the `knowledge_statement` to be clear and self-contained. Avoid including entire paragraphs if a single preceding sentence or phrase is sufficient.
4.  If the `knowledge_statement` is already self-contained, just return the `knowledge_statement` itself.
5.  Return only the final text, with no extra explanations.

# Example:
`subsection_text`: "The Transformer architecture has been very successful. It relies on a self-attention mechanism. This mechanism allows the model to weigh the importance of different words in the input sequence. For example, in the sentence 'The cat sat on the mat', self-attention can help the model understand that 'sat' is related to 'cat' and 'mat'."
`knowledge_statement`: "This mechanism allows the model to weigh the importance of different words in the input sequence."

Output: "The Transformer architecture has been very successful. It relies on a self-attention mechanism. This mechanism allows the model to weigh the importance of different words in the input sequence."

# Task:
Now, process the following.
"""

def query_for_context(subsection_text, raw_knowledge_statement):
    if not isinstance(subsection_text, str) or not isinstance(raw_knowledge_statement, str):
        return None
    prompt = {
        'system': context_prompt,
        'user': f"`subsection_text`:\n{subsection_text}\n\n`knowledge_statement`:\n{raw_knowledge_statement}"
    }
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=2048)

def check_statement_in_paper(statement):
    if not statement:
        return False
    statement_cleaned = statement.strip().strip('"`')
    return statement_cleaned.lower() in paper.lower()

inputs = [(row['subsection_text'], row['raw_knowledge_statement']) for _, row in paper_df_validated.iterrows()]

print(f"Adding context to {len(inputs)} knowledge statements...")
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(query_for_context, sub_text, raw_stmt) for sub_text, raw_stmt in inputs]
    statements_with_context = [future.result() for future in futures]

paper_df_validated['raw_knowledge_statement_with_context'] = statements_with_context

# Check which statements are found in paper
found_in_paper = [check_statement_in_paper(stmt) for stmt in statements_with_context]

# Retry failed statements twice
for retry in range(2):
    failed_indices = [i for i, found in enumerate(found_in_paper) if not found]
    if not failed_indices:
        break
    
    print(f"Retry {retry + 1}: Processing {len(failed_indices)} failed statements...")
    retry_inputs = [inputs[i] for i in failed_indices]
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        futures = [executor.submit(query_for_context, sub_text, raw_stmt) for sub_text, raw_stmt in retry_inputs]
        retry_results = [future.result() for future in futures]
    
    for i, result in zip(failed_indices, retry_results):
        statements_with_context[i] = result
        found_in_paper[i] = check_statement_in_paper(result)

# Update dataframe with final results
paper_df_validated['raw_knowledge_statement_with_context'] = statements_with_context

# Add context_needed column
paper_df_validated['context_needed'] = [
    stmt_with_context.strip() != raw_stmt.strip() 
    for stmt_with_context, raw_stmt in zip(statements_with_context, paper_df_validated['raw_knowledge_statement'])
]

# Filter out statements not found in paper
initial_rows = len(paper_df_validated)
paper_df_validated = paper_df_validated[found_in_paper]
final_rows = len(paper_df_validated)

print(f"Filtered out {initial_rows - final_rows} rows that were not found in the original paper text.")
print(f"Remaining rows: {final_rows}")

paper_df_validated

Adding context to 204 knowledge statements...
Retry 1: Processing 3 failed statements...
Retry 2: Processing 3 failed statements...
Filtered out 2 rows that were not found in the original paper text.
Remaining rows: 202


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,knowledge_statement,raw_knowledge_statement_with_context,context_needed
0,No Section,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Direct Preference Optimization: Your Language ...,\title{Direct Preference Optimization: Your La...,False
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,False
2,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining such steerability...,While large-scale unsupervised language models...,True
3,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...","However, RLHF is a complex and often unstable ...",Existing methods for gaining such steerability...,True
4,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,In this paper we introduce a new parameterizat...,"However, RLHF is a complex and often unstable ...",True
...,...,...,...,...,...,...,...,...,...
199,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...","For example, can training with self-labeling f...",Our initial results suggest that DPO policies ...,True
200,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...","On another front, how does reward over-optimiz...",Our initial results suggest that DPO policies ...,True
201,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...","Additionally, while we evaluate models up to 6...","On another front, how does reward over-optimiz...",True
202,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...","Regarding evaluations, we find that the win ra...","Additionally, while we evaluate models up to 6...",True


In [51]:
# print out the rows where context_needed is True. print the raw_knowledge_statement and the raw_knowledge_statement_with_context
context_needed_rows = paper_df_validated[paper_df_validated['context_needed'] == True]

for idx, row in context_needed_rows.iterrows():
    print(f"Row {idx}:")
    print(f"Raw statement: {row['raw_knowledge_statement']}")
    print(f"Statement with context: {row['raw_knowledge_statement_with_context']}")
    print("-" * 80)
    

Row 2:
Raw statement: Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
Statement with context: While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
--------------------------------------------------------------------------------
Row 3:
Raw statement: However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuni

### [NOT USED] Also clean the raw_knowledge statments with context

In [52]:
import concurrent.futures
# --- NEW: Clean unnecessary LaTeX from knowledge statements ---
cleaning_prompt = r"""Your task is to remove unmeaningful LaTeX commands while keeping the sentence same otherwise.

# Rules:
1.  **Keep Math**: Preserve mathematical notation written in LaTeX (e.g., `$\mathcal{L}$`, `\pi_\theta`, `\mathbb{E}`). Do not expand or remove them.
2.  **Remove Formatting**: Remove formatting commands like `\textbf{...}`, `\textit{...}`, `\texttt{...}`, etc., but keep the text inside them.
3.  **Handle Citations & References**: Simplify citation commands (e.g., `\cite{...}`, `\citep{...}`) and reference commands (e.g., `\ref{...}`) by removing them but keeping the sentence flow. For example, 'See Appendix~\ref{app:1}' becomes 'See Appendix'.
4.  **Keep Core Content**: The final output must be the original sentence otherwise.
5.  Return only the cleaned sentence, with no extra explanations.

# Example 1:
Input: `Our experiments show that \textbf{DPO} can fine-tune LMs to align with human preferences as well as or better than existing methods.`
Output: `Our experiments show that DPO can fine-tune LMs to align with human preferences as well as or better than existing methods.`

# Example 2:
Input: `The gradient with respect to the parameters $\theta$ can be written as:`
Output: `The gradient with respect to the parameters $\theta$ can be written as:`

# Example 3:
Input: `See Appendix~\ref{app:derivation1} for a complete derivation.`
Output: `See Appendix for a complete derivation.`

# Task:
Now, process the following sentence.
"""

def query_single_clean(statement):
    if statement is None:
        return None
    """Query the LLM to clean a single knowledge statement."""
    prompt = {
        'system': cleaning_prompt,
        'user': f"Input: `{statement}`"
    }
    # Assuming 'gpt-4.1' is a valid model alias in your utils
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=2048)

paper_df_validated.dropna(inplace=True)
# Get unique non-null statements to avoid redundant API calls
statements_to_clean = paper_df_validated['raw_knowledge_statement_with_context'].unique().tolist()
cleaned_statements = []

print(f"Cleaning {len(statements_to_clean)} unique knowledge statements...")
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(query_single_clean, stmt) for stmt in statements_to_clean]
    cleaned_statements = [future.result() for future in futures]

# Create a mapping from raw statement to cleaned statement
cleaning_map = dict(zip(statements_to_clean, cleaned_statements))

# Map the cleaned statements back to the dataframe
paper_df_validated['knowledge_statement_with_context'] = paper_df_validated['raw_knowledge_statement_with_context'].map(cleaning_map)

paper_df_validated.reset_index(drop=True, inplace=True)
paper_df_validated

Cleaning 202 unique knowledge statements...


/var/folders/_c/yxhjygdn79j4bxrfk18sjync0000gn/T/ipykernel_2905/890636045.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  paper_df_validated.dropna(inplace=True)
/var/folders/_c/yxhjygdn79j4bxrfk18sjync0000gn/T/ipykernel_2905/890636045.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  paper_df_validated['knowledge_statement_with_context'] = paper_df_validated['raw_knowledge_statement_with_context'].map(cleaning_map)


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,knowledge_statement,raw_knowledge_statement_with_context,context_needed,knowledge_statement_with_context
0,No Section,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Direct Preference Optimization: Your Language ...,\title{Direct Preference Optimization: Your La...,False,Direct Preference Optimization: Your Language ...
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,False,While large-scale unsupervised language models...
2,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining such steerability...,While large-scale unsupervised language models...,True,While large-scale unsupervised language models...
3,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...","However, RLHF is a complex and often unstable ...",Existing methods for gaining such steerability...,True,Existing methods for gaining such steerability...
4,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,In this paper we introduce a new parameterizat...,"However, RLHF is a complex and often unstable ...",True,"However, RLHF is a complex and often unstable ..."
...,...,...,...,...,...,...,...,...,...,...
197,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...","For example, can training with self-labeling f...",Our initial results suggest that DPO policies ...,True,Our initial results suggest that DPO policies ...
198,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...","On another front, how does reward over-optimiz...",Our initial results suggest that DPO policies ...,True,Our initial results suggest that DPO policies ...
199,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...","Additionally, while we evaluate models up to 6...","On another front, how does reward over-optimiz...",True,"On another front, how does reward over-optimiz..."
200,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...","Regarding evaluations, we find that the win ra...","Additionally, while we evaluate models up to 6...",True,"Additionally, while we evaluate models up to 6..."


### [OLD] Create atomic self-contained version of the text

In [14]:
# Create atomic self-contained versions of knowledge statements
atomic_prompt = r"""You are tasked with creating atomic, self-contained versions of knowledge statements. Given a knowledge statement and its surrounding paragraph context, you need to:

1. *Make it self-contained and explicit*: Add any necessary context from the paragraph so the statement can be understood independently. Only add what's necessary and keep it minimal.
2. *Keep it atomic*: Ensure it's coherent and unambiguous.
3. *Preserve the core meaning*: Stick to the information provided in the sentence as much as possible.
4. *Plain text when possible*: Do not include Latex tags such as \ref or \cite.
5. *Make sure equations are in Latex format*: Do not include equations in plain text. Use the same latex style as used in the original paragraph.
6. *Write in paper's style*: Use the same diction, linguistic structure, and syntax as the original knowledge statement when possible.

# Demonstrations

## Example 1:
Knowledge Statement: "This approach eliminates the need for reward model training by directly optimizing on preference data." # by directly optimizing on preference data.
Paragraph Context: "Direct Preference Optimization (DPO) is a novel method for training language models. Unlike RLHF, this approach eliminates the need for reward model training by directly optimizing on preference data."

Output: "Direct Preference Optimization (DPO) eliminates the need for reward model training by directly optimizing on preference data."
The original knowledge statement has a reference to "This approach" which is not self-contained, and so we make the reference explicit.

## Example 2:
Knowledge Statement: "The gradient can be computed efficiently without requiring Monte Carlo sampling."
Paragraph Context: "In our mathematical formulation of DPO, we derive a closed-form solution for the optimization objective. The gradient can be computed efficiently without requiring Monte Carlo sampling."

Output: "The gradient of the DPO optimization objective can be computed efficiently without requiring Monte Carlo sampling."
The original knowledge statement has a reference to "The gradient" which is not ambiguous, and so we make the information explicit.

# Task:
Create an atomic, self-contained version of the given knowledge statement using the paragraph context. ONLY return the atomic version of the knowledge statement, no other text.
"""

def create_atomic_statement(knowledge_statement, paragraph):
    if pd.isna(knowledge_statement) or pd.isna(paragraph):
        return None
    
    prompt = {
        'system': atomic_prompt,
        'user': f"Knowledge Statement: {knowledge_statement}\nParagraph Context: {paragraph}"
    }
    return utils.query_llm(prompt, model='gpt-4.1', temperature=0, max_tokens=1024)

# Create atomic versions
print(f"Creating atomic self-contained versions for {len(paper_df_validated)} statements...")
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    futures = [
        executor.submit(create_atomic_statement, row['raw_knowledge_statement'], row['paragraph'])
        for _, row in paper_df_validated.iterrows()
    ]
    atomic_statements = [future.result() for future in futures]

paper_df_validated['atomic_knowledge_statement'] = atomic_statements
paper_df_validated


Creating atomic self-contained versions for 204 statements...


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,atomic_knowledge_statement
0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining steerability in l...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...",Reinforcement learning from human feedback (RL...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,"In this paper, we introduce a new parameteriza..."
...,...,...,...,...,...,...,...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...",An open question is whether training with self...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...",How does reward over-optimization manifest in ...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...",While we evaluate DPO on models up to 6B param...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...",We find that the win rates computed by GPT-4 d...


In [15]:

for i in range(100):
    print(paper_df_validated['atomic_knowledge_statement'].iloc[i])
    print(paper_df_validated['raw_knowledge_statement'].iloc[i])
    print("----")


\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}
\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}
----
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of the behavior of these models is difficult due to the completely unsupervised nature of their training.
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
----
Existing methods for gaining steerability in large-scale unsupervised language models collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often using reinforcement learning from human feedback (RLHF).
Existing methods for gaining such steerability collect human labels 

In [ ]:
# paper_df_validated.to_csv('../../data/arxiv/DPO_knowledge_probes_v3.csv', index=False)

### Identify Target Span Given the Piece of Knowledge

#### [NOT USED]

In [22]:
# Define the prompt for extracting target spans
target_span_prompt = """You are tasked with identifying the last K sentences from a knowledge statement that can serve as a target span for testing language model knowledge.

Your goal is to extract the final portion of the knowledge statement that:
1. Contains the most specific, testable information
2. Can be masked/removed to create a meaningful knowledge probe

# Instructions:
- Extract the last 1-5 words from the knowledge statement
- Make sure the span is the last part of the sentence and not in the middle of the sentence
- If the span includes equations, it's okay to include latex commands and try to appropriately extract part of the equation (it will probably be longer than 5 words)
- The span should be the "punchline" or main conclusion of the statement
- You should not include function words like "and", "of", or "in" 
- If the last few words are really useless english, at least try to find the target near the end of the sentence
- Return only extracted words, no other text
"""

def extract_target_span(knowledge_statement):
    if pd.isna(knowledge_statement):
        return None
    
    prompt = {
        'system': target_span_prompt,
        'user': f"Knowledge Statement: {knowledge_statement}"
    }
    return utils.query_llm(prompt, model='gpt-4.1-mini', temperature=0, max_tokens=512)

# Extract target spans for knowledge probes
print(f"Extracting target spans for {len(paper_df_validated)} statements...")
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = [
        executor.submit(extract_target_span, row['knowledge_statement_with_context'])
        for _, row in paper_df_validated.iterrows()
    ]
    target_spans = [future.result() for future in futures]

paper_df_validated['target_span'] = target_spans
paper_df_validated


Extracting target spans for 210 statements...


KeyError: 'knowledge_statement_with_context'

In [62]:
for i in range(100):
    print(paper_df_validated['knowledge_statement_with_context'].iloc[i])
    print(paper_df_validated['target_span'].iloc[i])
    print("----")

Direct Preference Optimization: Your Language Model is Secretly a Reward Model
Secretly a Reward Model
----
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
completely unsupervised nature of their training
----
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
reinforcement learning from human feedback (RLHF).
----
Existing methods for gaining such steerability collect human labels of the relative qual

In [23]:
# Create knowledge probes by removing target spans from knowledge statements
def create_knowledge_probe(knowledge_statement, target_span):
    if pd.isna(knowledge_statement) or pd.isna(target_span):
        return None
    
    # Strip ending punctuation from target span
    target_span_clean = target_span.rstrip('.,!?;:')
    
    # Split by target span and take the before portion
    if target_span_clean in knowledge_statement:
        probe = knowledge_statement.split(target_span_clean)[0]
        return probe
    
    return None

# Create knowledge probes for each row
knowledge_probes = []
for _, row in paper_df_validated.iterrows():
    probe = create_knowledge_probe(row['knowledge_statement_with_context'], row['target_span'])
    knowledge_probes.append(probe)

paper_df_validated['knowledge_probe_with_context'] = knowledge_probes
paper_df_validated


KeyError: 'knowledge_statement_with_context'

#### New

In [16]:
# Define the prompt for extracting target spans
target_span_prompt = """You are tasked with identifying the last 1-2 words from a knowledge statement that can serve as a target span for testing language model knowledge.

Your goal is to extract the final portion of the knowledge statement that:
1. Contains the most specific, testable information
2. Can be masked/removed to create a meaningful knowledge probe

# Instructions
- The target span MUST be from the end of the sentence.
- The target span must not be more than two words (if it's an equation, it should be some monomial term).
- The target span should be the "punchline" of the statement. It should be meaningful such that it actually tests for some sort of knowledge.
- You should not include words like "and", "of", or "in" or acronyms like "(RLHF)" when the target includes "Reinforcement Learning from Human Feedback".
- The target span should NOT include latex references like \ref{sec:method} or \cite{smith2023}.
- But, if the span includes equations, it's okay to include latex commands and appropriately extract an atomic part of the equation.
- The target span can include numerical details like "100" or "0.01".
- If a valid target span doesn't exist (e.g. the target is not meaningful or too long), return "None"
- Return only extracted words, no other text.

# Examples

Knowledge Statement: "Large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, but achieving precise control of their behavior is difficult because their training is completely unsupervised."
Target Span: "their training is completely unsupervised"

Knowledge Statement: "Large unsupervised language models are trained on data generated by humans who have a wide variety of goals, priorities, and skillsets." 
Target Span: "goals, priorities, and skillsets"
"""

def extract_target_span(knowledge_statement):
    if pd.isna(knowledge_statement):
        return None
    
    prompt = {
        'system': target_span_prompt,
        'user': f"Knowledge Statement: {knowledge_statement}"
    }
    return utils.query_llm(prompt, model='gpt-4.1-mini', temperature=0, max_tokens=512)

# Extract target spans for knowledge probes
print(f"Extracting target spans for {len(paper_df_validated)} statements...")
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = [
        executor.submit(extract_target_span, row['atomic_knowledge_statement'])
        for _, row in paper_df_validated.iterrows()
    ]
    target_spans = [future.result() for future in futures]

paper_df_validated['atomic_target_span'] = target_spans
paper_df_validated


Extracting target spans for 204 statements...


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,atomic_knowledge_statement,atomic_target_span
0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,None
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,their training
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining steerability in l...,human feedback
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...",Reinforcement learning from human feedback (RL...,original model
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,"In this paper, we introduce a new parameteriza...",classification loss
...,...,...,...,...,...,...,...,...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...",An open question is whether training with self...,unlabeled prompts
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...",How does reward over-optimization manifest in ...,reward over-optimization
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...",While we evaluate DPO on models up to 6B param...,future work
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...",We find that the win rates computed by GPT-4 d...,high-quality judgments


In [17]:
# Create knowledge probes by removing target spans from knowledge statements
def create_knowledge_probe(knowledge_statement, target_span):
    if pd.isna(knowledge_statement) or pd.isna(target_span):
        return None
    
    # Strip ending punctuation from target span
    target_span_clean = target_span.rstrip('.,!?;:')
    
    # Split by target span and take the before portion
    if target_span_clean in knowledge_statement:
        probe = knowledge_statement.split(target_span_clean)[0]
        return probe
    
    return None

# Create knowledge probes for each row
knowledge_probes = []
for _, row in paper_df_validated.iterrows():
    probe = create_knowledge_probe(row['atomic_knowledge_statement'], row['atomic_target_span'])
    knowledge_probes.append(probe)

paper_df_validated['atomic_knowledge_probe'] = knowledge_probes

# Drop rows where atomic_knowledge_probe is None
initial_count = len(paper_df_validated)
paper_df_with_probes = paper_df_validated.dropna(subset=['atomic_knowledge_probe']).reset_index(drop=True)
final_count = len(paper_df_with_probes)
dropped_count = initial_count - final_count

print(f"Dropped {dropped_count} rows where atomic_knowledge_probe is None")
print(f"Remaining rows: {final_count}")

paper_df_with_probes


Dropped 6 rows where atomic_knowledge_probe is None
Remaining rows: 198


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,atomic_knowledge_statement,atomic_target_span,atomic_knowledge_probe
0,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,their training,While large-scale unsupervised language models...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining steerability in l...,human feedback,Existing methods for gaining steerability in l...
2,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...",Reinforcement learning from human feedback (RL...,original model,Reinforcement learning from human feedback (RL...
3,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,"In this paper, we introduce a new parameteriza...",classification loss,"In this paper, we introduce a new parameteriza..."
4,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"The resulting algorithm, which we call \textit...",Direct Preference Optimization (DPO) is a stab...,hyperparameter tuning,Direct Preference Optimization (DPO) is a stab...
...,...,...,...,...,...,...,...,...,...
193,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...",An open question is whether training with self...,unlabeled prompts,An open question is whether training with self...
194,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...",How does reward over-optimization manifest in ...,reward over-optimization,How does
195,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...",While we evaluate DPO on models up to 6B param...,future work,While we evaluate DPO on models up to 6B param...
196,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...",We find that the win rates computed by GPT-4 d...,high-quality judgments,We find that the win rates computed by GPT-4 d...


##### Old  targets

In [27]:
for i in range(191):
    print_wrapped(paper_df_with_probes['atomic_knowledge_statement'].iloc[i])
    print("-")
    print_wrapped(paper_df_with_probes['atomic_target_span'].iloc[i])
    print("---")

While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning
skills, achieving precise control of their behavior is difficult due to the completely unsupervised
nature of their training.

-
completely unsupervised nature of their training

---
Existing methods for gaining steerability in large-scale unsupervised language models collect human
labels of the relative quality of model generations and fine-tune the unsupervised LM to align with
these preferences, often using reinforcement learning from human feedback (RLHF).

-
reinforcement learning from human feedback

---
Reinforcement learning from human feedback (RLHF) is a complex and often unstable procedure, first
fitting a reward model that reflects human preferences, and then fine-tuning the large unsupervised
language model (LM) using reinforcement learning to maximize this estimated reward without drifting
too far from the original model.

-
the original model

---
In this paper, we introdu

##### New  targets

In [18]:
for i in range(191):
    print_wrapped(paper_df_with_probes['atomic_knowledge_statement'].iloc[i])
    print("-")
    print_wrapped(paper_df_with_probes['atomic_target_span'].iloc[i])
    print("---")

While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning
skills, achieving precise control of the behavior of these models is difficult due to the completely
unsupervised nature of their training.

-
their training

---
Existing methods for gaining steerability in large-scale unsupervised language models collect human
labels of the relative quality of model generations and fine-tune the unsupervised LM to align with
these preferences, often using reinforcement learning from human feedback (RLHF).

-
human feedback

---
Reinforcement learning from human feedback (RLHF) is a complex and often unstable procedure, first
fitting a reward model that reflects human preferences, and then fine-tuning the large unsupervised
language model (LM) using reinforcement learning to maximize this estimated reward without drifting
too far from the original model.

-
original model

---
In this paper, we introduce a new parameterization of the reward model in rei

Ensure tokenizing the target separately from the probe is fine

In [32]:
# Clean up atomic_knowledge_probe (remove trailing space) and atomic_target_span (add leading space)
paper_df_with_probes['atomic_knowledge_probe'] = paper_df_with_probes['atomic_knowledge_probe'].str.rstrip()

# Add leading space and handle period at end of atomic_knowledge_statement
updated_target_spans = []
period_added_count = 0
stripped_and_period_added_count = 0
validation_errors = 0

for _, row in paper_df_with_probes.iterrows():
    target_span = str(row['atomic_target_span'])
    knowledge_statement = str(row['atomic_knowledge_statement'])
    probe = str(row['atomic_knowledge_probe'])
    
    # Check if target_span + "." is at the end of the knowledge statement
    if knowledge_statement.endswith(target_span + "."):
        target_span = target_span + "."
        period_added_count += 1
    elif knowledge_statement.rstrip().endswith(target_span + "."):
        target_span = target_span + "."
        stripped_and_period_added_count += 1
    
    # Add leading space to target span
    final_target_span = ' ' + target_span.lstrip()
    updated_target_spans.append(final_target_span)
    
    # Validate that probe + target_span exists in knowledge_statement
    reconstructed = probe + final_target_span
    if reconstructed not in knowledge_statement:
        validation_errors += 1
        print(f"❌ Validation error for row {_}: probe + target not found in knowledge statement")
        print(f"  Probe: '{probe}'")
        print(f"  Target: '{final_target_span}'")
        print(f"  Reconstructed: '{reconstructed}'")
        print(f"  Knowledge statement: '{knowledge_statement}'")
        print()

paper_df_with_probes['atomic_target_span'] = updated_target_spans

print(f"Cleaned up atomic_knowledge_probe (removed trailing spaces) and atomic_target_span (added leading space and period when appropriate)")
print(f"Added period to {period_added_count} target spans")
print(f"Added period to {stripped_and_period_added_count} target spans")
print(f"Validation errors: {validation_errors}")


❌ Validation error for row 71: probe + target not found in knowledge statement
  Probe: 'Analogous to the reward modeling approach, the maximum likelihood objective for a parametrized policy $\pi_\theta$ becomes:
\begin{equation}\label{eq:optimum_model}'
  Target: ' \mathcal{L}_\text{DPO}(\pi_{\theta}; \piref)'
  Reconstructed: 'Analogous to the reward modeling approach, the maximum likelihood objective for a parametrized policy $\pi_\theta$ becomes:
\begin{equation}\label{eq:optimum_model} \mathcal{L}_\text{DPO}(\pi_{\theta}; \piref)'
  Knowledge statement: 'Analogous to the reward modeling approach, the maximum likelihood objective for a parametrized policy $\pi_\theta$ becomes:
\begin{equation}\label{eq:optimum_model}
    \mathcal{L}_\text{DPO}(\pi_{\theta}; \piref) = -\mathbb{E}_{(x, y_w, y_l)\sim \mathcal{D}}\left[\log \sigma \left(\beta \log \frac{\pi_{\theta}(y_w\mid x)}{\piref(y_w\mid x)} - \beta \log \frac{\pi_{\theta}(y_l\mid x)}{\piref(y_l\mid x)}\right)\right].
\end{equatio

In [138]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B")

contexts = paper_df_with_probes['atomic_knowledge_probe'].tolist()
targets = paper_df_with_probes['atomic_target_span'].tolist()

print(f"--- Tokenizer Sanity Check ---")

mismatches = 0
for i, (context, target) in enumerate(zip(contexts, targets)):
    # --- Tokenization ---
    # Method 1: Tokenize the fully concatenated string.
    whole_string = context + target
    tokenized_whole = tokenizer(whole_string, add_special_tokens=False)['input_ids']

    # Method 2: Tokenize parts separately and concatenate the resulting token ID lists.
    tokenized_context = tokenizer(context, add_special_tokens=False)['input_ids']
    tokenized_target = tokenizer(target, add_special_tokens=False)['input_ids']
    tokenized_parts_combined = tokenized_context + tokenized_target

    # --- Comparison ---
    if tokenized_whole != tokenized_parts_combined:
        mismatches += 1
        print(f"--- ❌ MISMATCH DETECTED: Sample #{i} ---")
        print(f"Context: '{context}'")
        print(f"Target:  '{target}'")
        print(f"\nTokenizing whole string:            {tokenized_whole} (Length: {len(tokenized_whole)})")
        print(f"Tokenizing parts and concatenating: {tokenized_parts_combined} (Length: {len(tokenized_parts_combined)})")
        
        # Find differing positions
        min_len = min(len(tokenized_whole), len(tokenized_parts_combined))
        diff_positions = []
        for pos in range(min_len):
            if tokenized_whole[pos] != tokenized_parts_combined[pos]:
                diff_positions.append(pos)
        
        if diff_positions:
            print(f"\nFirst differing positions: {diff_positions[:5]}")
            for pos in diff_positions[:3]:
                whole_token = tokenizer.decode([tokenized_whole[pos]])
                parts_token = tokenizer.decode([tokenized_parts_combined[pos]])
                print(f"  Position {pos}: whole='{whole_token}' vs parts='{parts_token}'")
        
        print("-" * 30)


--- Tokenizer Sanity Check ---


### Create paraphrased probes of atomic fact

In [2]:
import pandas as pd
import sys 
sys.path.append('../..')
import utils.utils as utils

paper_df_with_probes = pd.read_csv('../../data/arxiv/DPO_knowledge_probes_v1.csv')

paper_df_with_probes.head()

,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,knowledge_statement,atomic_knowledge_statement,atomic_target_span,atomic_knowledge_probe
0,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,Large-scale unsupervised language models (LMs)...,their training is completely unsupervised,Large-scale unsupervised language models (LMs)...
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining such steerability...,Existing methods for achieving precise control...,align with these preferences,Existing methods for achieving precise control...
2,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,In this paper we introduce a new parameterizat...,The new parameterization of the reward model i...,simple classification loss,The new parameterization of the reward model i...
3,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"The resulting algorithm, which we call \textit...","The resulting algorithm, which we call Direct ...",Direct Preference Optimization (DPO) is a stab...,significant hyperparameter tuning,Direct Preference Optimization (DPO) is a stab...
4,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Our experiments show that DPO can fine-tune LM...,Our experiments show that DPO can fine-tune LM...,Experiments demonstrate that Direct Preference...,simpler to implement and train,Experiments demonstrate that Direct Preference...


In [3]:
# Check and fix atomic_target_span endings
for i in range(len(paper_df_with_probes)):
    target_span = paper_df_with_probes.loc[i, 'atomic_target_span']
    if not target_span.endswith('.'):
        paper_df_with_probes.loc[i, 'atomic_target_span'] = target_span + '.'

print(f"Updated {len(paper_df_with_probes)} rows to ensure atomic_target_span ends with '.'")


Updated 189 rows to ensure atomic_target_span ends with '.'


In [14]:
paper_df_with_probes.iloc[66]


section                                          Direct Preference Optimization
subsection                                                        No Subsection
paragraph                     \textbf{Deriving the DPO objective.} We start ...
section_text                  \label{sec:DPO}\n\nMotivated by the challenges...
subsection_text               \label{sec:DPO}\n\nMotivated by the challenges...
raw_knowledge_statement       However, we can rearrange Eq.~\ref{eq:op_polic...
knowledge_statement           However, we can rearrange Eq. to express the r...
atomic_knowledge_statement    We can rearrange the expression for the optima...
atomic_target_span                                             \beta \log Z(x).
atomic_knowledge_probe        We can rearrange the expression for the optima...
Name: 66, dtype: object

In [8]:
import pandas as pd
from tqdm import tqdm
import asyncio
import aiohttp
from concurrent.futures import ThreadPoolExecutor
import threading

# It is assumed that a `query_llm` function is defined in your environment,
# which takes a prompt string and returns the LLM's response.
# For example:
#
# from some_llm_library import Llama
# llm = Llama.build(...)
# def query_llm(prompt):
#     return llm.text_completion(prompt)

def paraphrase_probe(probe, target, llm_querier):
    """
    Generates a paraphrased version of a knowledge probe using an LLM.

    Args:
        probe (str): The knowledge probe to paraphrase.
        target (str): The target span that the paraphrase must end with.
        llm_querier (function): A function that takes a prompt and returns the LLM's response.

    Returns:
        str: The paraphrased knowledge probe.
    """
    prompt = {
        "system": (
            f"Your task is to paraphrase the given sentence "
            f"while ensuring the paraphrased version is a single, complete sentence that ends with the "
            f"exact phrase: '{target.rstrip('.')}'. Note that the paraphrased sentence should be semantically equivalent to the original sentence, \
and it should not contain any additional factual knowledge, nor lacks any factual knowledge that is \
stated in the original text. Lastly, the paraphrased sentence MUST end with the exact phrase: '{target.rstrip('.')}'. Return only the paraphrased sentence, no other text."
        ),
        "user": (
            f"Sentence: '{probe}{target.rstrip('.')}.'\n\n"
            f"Target: '{target.rstrip('.')}':"
        )
    }
    
    paraphrased_full_sentence = llm_querier(prompt)
    print(paraphrased_full_sentence)
    # Clean up the response to ensure it's just the paraphrased sentence
    # and extract the new probe part.
    if paraphrased_full_sentence.endswith(target):
        paraphrased_probe = paraphrased_full_sentence[:-len(target)]
        return paraphrased_probe.strip()
    elif paraphrased_full_sentence.strip().endswith(target+"."):
        paraphrased_probe = paraphrased_full_sentence[:-len(target)-1]+"."
        return paraphrased_probe.strip()
    return f"Failed to paraphrase: {paraphrased_full_sentence}"

def generate_paraphrases_for_row(row, num_paraphrases=10):
    """Generate multiple unique paraphrases for a single row."""
    probe = row['atomic_knowledge_probe']
    target = row['atomic_target_span']
    
    paraphrases = set()
    max_rounds = 1  # Maximum number of rounds to try
    
    for round_num in range(max_rounds):
        if len(paraphrases) >= num_paraphrases:
            break
            
        needed = num_paraphrases - len(paraphrases)
        print(f"Round {round_num + 1}: Generating {needed} more paraphrases...")
        
        with ThreadPoolExecutor(max_workers=num_paraphrases) as executor:
            futures = [
                executor.submit(
                    paraphrase_probe, 
                    probe, 
                    target, 
                    lambda prompt: utils.query_llm(prompt, model="gpt-4.1-mini", temperature=1.25, top_p=0.95)
                )
                for _ in range(needed)
            ]
            
            for future in futures:
                result = future.result()
                if not result.startswith('Failed to paraphrase'):
                    paraphrases.add(result)
        print(len(paraphrases))
    return list(paraphrases)[:num_paraphrases]

# --- Test on one row first ---

# Test on just the first row
test_row = paper_df_with_probes.iloc[66]
print(f"Original probe: {test_row['atomic_knowledge_probe']}")
print(f"Target span: {test_row['atomic_target_span']}")
print("\nGenerating paraphrases...")

paraphrases = generate_paraphrases_for_row(test_row, num_paraphrases=10)

print(f"\nGenerated {len(paraphrases)} unique paraphrases:")
for i, paraphrase in enumerate(paraphrases):
    print(f"{i+1}. {paraphrase}")

Original probe: We can rearrange the expression for the optimal policy $\pi_r(y\mid x) = \frac{1}{Z(x)}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)$ to express the reward function $r(x, y)$ in terms of its corresponding optimal policy $\pi_r$, the reference policy $\piref$, and the unknown partition function $Z(x)$ as $r(x,y) =\beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)} +
Target span:  \beta \log Z(x).

Generating paraphrases...
Round 1: Generating 10 more paraphrases...
The reward function \(r(x, y)\) can be expressed in terms of the optimal policy \(\pi_r\), the reference policy \(\piref\), and the partition function \(Z(x)\) as \(r(x,y) = \beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)} + \beta \log Z(x).\)
The reward function \( r(x,y) \) can be written in terms of the optimal policy \( \pi_r \), the reference policy \( \piref \), and the partition function \( Z(x) \) as \( r(x,y) = \beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)} + \beta \log Z(x) \).
By rearrang

In [9]:
# Check tokenization of "Z(x)" variants
import transformers

# Load the tokenizer (using the same one as in your model)
tokenizer = transformers.AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B")

# Test strings
test_strings = ["Z(x)", "Z(x).", "Z(x) )", "\\beta \\log Z(x)"]

print("Checking tokenization of Z(x) variants:")
for test_str in test_strings:
    tokens = tokenizer.tokenize(test_str)
    token_ids = tokenizer.encode(test_str, add_special_tokens=False)
    print(f"'{test_str}' -> tokens: {tokens} -> ids: {token_ids}")

# Check if "Z(x)" is a common prefix
zx_tokens = tokenizer.tokenize("Z(x)")
zx_ids = tokenizer.encode("Z(x)", add_special_tokens=False)

print(f"\nBase 'Z(x)' tokens: {zx_tokens} -> ids: {zx_ids}")

for test_str in ["Z(x).", "Z(x) )"]:
    test_tokens = tokenizer.tokenize(test_str)
    test_ids = tokenizer.encode(test_str, add_special_tokens=False)
    
    # Check if Z(x) tokens are a prefix
    is_prefix = len(zx_ids) <= len(test_ids) and test_ids[:len(zx_ids)] == zx_ids
    print(f"'{test_str}' has 'Z(x)' as prefix: {is_prefix}")


Checking tokenization of Z(x) variants:
'Z(x)' -> tokens: ['Z', '(x', ')'] -> ids: [57, 2120, 8]
'Z(x).' -> tokens: ['Z', '(x', ').'] -> ids: [57, 2120, 570]
'Z(x) )' -> tokens: ['Z', '(x', ')', 'Ġ)'] -> ids: [57, 2120, 8, 883]
'\beta \log Z(x)' -> tokens: ['\\', 'beta', 'Ġ\\', 'log', 'ĠZ', '(x', ')'] -> ids: [59, 19674, 1144, 848, 1901, 2120, 8]

Base 'Z(x)' tokens: ['Z', '(x', ')'] -> ids: [57, 2120, 8]
'Z(x).' has 'Z(x)' as prefix: False
'Z(x) )' has 'Z(x)' as prefix: True


In [31]:
# Check tokenization of the mathematical expression
test_expression = r"\beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)}$."

print(f"Testing tokenization of: {test_expression}")
tokens = tokenizer.tokenize(test_expression)
token_ids = tokenizer.encode(test_expression, add_special_tokens=False)
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(tokens)}")


Testing tokenization of: \beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)}$.
Tokens: ['\\', 'beta', 'Ġ\\', 'log', 'Ġ\\', 'frac', '{\\', 'pi', '_r', '(y', '\\', 'mid', 'Ġx', ')}', '{\\', 'pire', 'f', '(y', '\\', 'mid', 'Ġx', ')}', '$.']
Token IDs: [59, 19674, 1144, 848, 1144, 38118, 36802, 2554, 1745, 7166, 59, 16497, 865, 9317, 36802, 23772, 69, 7166, 59, 16497, 865, 9317, 13244]
Number of tokens: 23


In [23]:
import pandas as pd
from tqdm import tqdm
import asyncio
import aiohttp
from concurrent.futures import ThreadPoolExecutor
import threading

# It is assumed that a `query_llm` function is defined in your environment,
# which takes a prompt string and returns the LLM's response.
# For example:
#
# from some_llm_library import Llama
# llm = Llama.build(...)
# def query_llm(prompt):
#     return llm.text_completion(prompt)

def paraphrase_probe(probe, target, llm_querier):
    """
    Generates a paraphrased version of a knowledge probe using an LLM.

    Args:
        probe (str): The knowledge probe to paraphrase.
        target (str): The target span that the paraphrase must end with.
        llm_querier (function): A function that takes a prompt and returns the LLM's response.

    Returns:
        str: The paraphrased knowledge probe.
    """
    prompt = {
        "system": (
            f"You are an expert at paraphrasing sentences. Your task is to paraphrase the given sentence "
            f"while ensuring the paraphrased version is a single, complete sentence that ends with the "
            f"exact phrase: '{target}'. Again, the paraphrased sentence MUST end with the exact phrase: '{target}'. Return only the paraphrased sentence, no other text."
        ),
        "user": (
            f"Original sentence: '{probe}{target}'\n\n"
            f"Paraphrased sentence ending with '{target}':"
        )
    }
    
    paraphrased_full_sentence = llm_querier(prompt)
    
    # Clean up the response to ensure it's just the paraphrased sentence
    # and extract the new probe part.
    if paraphrased_full_sentence.endswith(target):
        paraphrased_probe = paraphrased_full_sentence[:-len(target)]
        return paraphrased_probe.strip()
    elif paraphrased_full_sentence.rstrip(".").endswith(target):
        paraphrased_probe = paraphrased_full_sentence[:-len(target) - 1]
        return paraphrased_probe.strip()
    return f"Failed to paraphrase: {paraphrased_full_sentence}"

def generate_paraphrases_for_row(row, num_paraphrases=10):
    """Generate multiple unique paraphrases for a single row."""
    probe = row['atomic_knowledge_probe']
    target = row['atomic_target_span']
    
    paraphrases = set()
    max_rounds = 10  # Maximum number of rounds to try
    
    for round_num in range(max_rounds):
        if len(paraphrases) >= num_paraphrases:
            break
            
        needed = num_paraphrases - len(paraphrases)
        
        with ThreadPoolExecutor(max_workers=num_paraphrases) as executor:
            futures = [
                executor.submit(
                    paraphrase_probe, 
                    probe, 
                    target, 
                    lambda prompt: utils.query_llm(prompt, model="gpt-4.1-mini", temperature=1.5, top_p=0.975)
                )
                for _ in range(needed)
            ]
            
            for future in futures:
                result = future.result()
                if not result.startswith("Failed to paraphrase:"):
                    paraphrases.add(result)
    if len(paraphrases) < num_paraphrases:
        print(f"Warning: Only {len(paraphrases)} paraphrases generated for row {row.name}")
    return list(paraphrases)[:num_paraphrases]

# --- Main Execution ---

# Initialize tqdm for progress tracking with pandas
tqdm.pandas(desc="Paraphrasing probes")

# Apply the paraphrasing function to each row of the DataFrame
paper_df_with_probes['paraphrased_atomic_knowledge_probes'] = paper_df_with_probes.progress_apply(
    generate_paraphrases_for_row,
    axis=1
)

# Display the DataFrame with the new column
print(paper_df_with_probes[['atomic_knowledge_probe', 'atomic_target_span', 'paraphrased_atomic_knowledge_probes']].head())

Paraphrasing probes:  15%|█▍        | 28/189 [01:07<15:37,  5.82s/it]

Paraphrasing probes:  36%|███▌      | 68/189 [03:50<32:59, 16.36s/it]

Paraphrasing probes:  47%|████▋     | 88/189 [05:06<10:42,  6.36s/it]

Paraphrasing probes:  51%|█████▏    | 97/189 [05:58<13:52,  9.04s/it]

Paraphrasing probes:  58%|█████▊    | 110/189 [06:45<07:58,  6.05s/it]

Paraphrasing probes:  60%|██████    | 114/189 [07:10<08:34,  6.86s/it]

Paraphrasing probes:  82%|████████▏ | 155/189 [09:45<05:55, 10.45s/it]

Paraphrasing probes:  86%|████████▌ | 163/189 [10:14<01:38,  3.77s/it]


KeyboardInterrupt: 

In [24]:
paper_df_with_probes.loc[66]

section                                                   Direct Preference Optimization
subsection                                                                 No Subsection
paragraph                              \textbf{Deriving the DPO objective.} We start ...
section_text                           \label{sec:DPO}\n\nMotivated by the challenges...
subsection_text                        \label{sec:DPO}\n\nMotivated by the challenges...
raw_knowledge_statement                However, we can rearrange Eq.~\ref{eq:op_polic...
knowledge_statement                    However, we can rearrange Eq. to express the r...
atomic_knowledge_statement             We can rearrange the expression for the optima...
atomic_target_span                                                       \beta \log Z(x)
atomic_knowledge_probe                 We can rearrange the expression for the optima...
paraphrased_atomic_knowledge_probes                                                   []
Name: 66, dtype: obje

In [ ]:
# Check the length of paraphrased_atomic_knowledge_probes lists
probe_lengths = paper_df_with_probes['paraphrased_atomic_knowledge_probes'].apply(len)
print("Length distribution of paraphrased_atomic_knowledge_probes:")
print(probe_lengths.value_counts().sort_index())
print(f"\nRows with less than 10 paraphrases: {(probe_lengths < 10).sum()}")
print(f"Rows with exactly 10 paraphrases: {(probe_lengths == 10).sum()}")

paper_df_with_probes

Length distribution of paraphrased_atomic_knowledge_probes:
paraphrased_atomic_knowledge_probes
0       1
2       1
4       1
5       1
6       3
7       2
8       5
9       9
10    166
Name: count, dtype: int64

Rows with less than 10 paraphrases: 23
Rows with exactly 10 paraphrases: 166


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,knowledge_statement,atomic_knowledge_statement,atomic_target_span,atomic_knowledge_probe,paraphrased_atomic_knowledge_probes
0,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,While large-scale unsupervised language models...,Large-scale unsupervised language models (LMs)...,their training is completely unsupervised,Large-scale unsupervised language models (LMs)...,[Although large-scale unsupervised language mo...
1,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,Existing methods for gaining such steerability...,Existing methods for achieving precise control...,align with these preferences,Existing methods for achieving precise control...,[Current approaches to precisely steer large-s...
2,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,In this paper we introduce a new parameterizat...,The new parameterization of the reward model i...,simple classification loss,The new parameterization of the reward model i...,[This paper presents a new parameterization of...
3,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"The resulting algorithm, which we call \textit...","The resulting algorithm, which we call Direct ...",Direct Preference Optimization (DPO) is a stab...,significant hyperparameter tuning,Direct Preference Optimization (DPO) is a stab...,[Direct Preference Optimization (DPO) is a rel...
4,No Section,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Our experiments show that DPO can fine-tune LM...,Our experiments show that DPO can fine-tune LM...,Experiments demonstrate that Direct Preference...,simpler to implement and train,Experiments demonstrate that Direct Preference...,[Experiments show that Direct Preference Optim...
...,...,...,...,...,...,...,...,...,...,...,...
184,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","For example, can training with self-labeling f...","For example, can training with self-labeling f...",A key open question is whether training with s...,unlabeled prompts,A key open question is whether training with s...,[An important unresolved question is whether u...
185,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","On another front, how does reward over-optimiz...","On another front, how does reward over-optimiz...",It is an open question how reward over-optimiz...,slight decrease in performance,It is an open question how reward over-optimiz...,[How reward over-optimization appears in the d...
186,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Additionally, while we evaluate models up to 6...","Additionally, while we evaluate models up to 6...",While the current evaluation of Direct Prefere...,orders of magnitude larger,While the current evaluation of Direct Prefere...,[Although the current assessment of Direct Pre...
187,Discussion,No Subsection,\textbf{Limitations \& Fut

In [20]:
paper_df_with_probes.to_csv("../../data/arxiv/DPO_knowledge_probes_v2.csv", index=False)

### Create QAs from the raw statement knowledge

In [15]:
def generate_qa_for_statement(statement, paragraph_context, llm_querier):
    """
    Generates 1-3 question-answer pairs that test the knowledge in the statement.

    Args:
        statement (str): The knowledge statement to create QAs for.
        paragraph_context (str): The paragraph context for the statement.
        llm_querier (function): A function that takes a prompt and returns the LLM's response.

    Returns:
        list: List of dictionaries with 'question' and 'answer' keys.
    """
    prompt = {
        "system": (
            "You are an expert at creating question-answer pairs that sharply test the key information at hand. Your task is to generate 1-3 "
            "question-answer pairs that test the knowledge contained in the given statement. "
            "The questions should be clear, specific, and directly testable from the statement. "
            "IMPORTANT: Each answer must be exactly one or two words only. No longer answers. "
            "Return your response as a JSON list of objects with 'question' and 'answer' keys. "
            "Example format: [{'question': 'What algorithm optimizes preferences directly?', 'answer': 'DPO'}]"
        ),
        "user": (
            f"Paragraph context: '{paragraph_context}'\n\n"
            f"Knowledge statement: '{statement}'\n\n"
            f"Generate 1-3 question-answer pairs that test this knowledge. Remember: answers must be 1-2 words only:"
        )
    }
    
    response = llm_querier(prompt)
    
    try:
        import json
        qa_pairs = json.loads(response.strip())
        return qa_pairs if isinstance(qa_pairs, list) else []
    except:
        return []

def generate_qas_for_row(row, max_qas=3):
    """Generate QA pairs for a single row's knowledge statement."""
    statement = row['atomic_knowledge_statement']
    paragraph = row['paragraph']
    
    try:
        qa_pairs = generate_qa_for_statement(
            statement, 
            paragraph,
            lambda prompt: utils.query_llm(prompt, model="gpt-4.1-mini", temperature=0.7)
        )
        return qa_pairs[:max_qas]  # Limit to max_qas
    except Exception as e:
        print(f"Failed to generate QAs for row {row.name}: {e}")
        return []

# Test on just one row first
test_row = paper_df_with_probes.iloc[0]
print(f"Testing QA generation on row 0:")
print(f"Statement: {test_row['atomic_knowledge_statement']}")
print(f"Paragraph: {test_row['paragraph'][:100]}...")

test_qa_pairs = generate_qas_for_row(test_row)
print(f"Generated QA pairs: {test_qa_pairs}")


NameError: name 'paper_df_with_probes' is not defined

In [ ]:
def generate_qa_for_statement(statement, paragraph_context, llm_querier):
    """
    Generates 1-3 question-answer pairs that test the knowledge in the statement.

    Args:
        statement (str): The knowledge statement to create QAs for.
        paragraph_context (str): The paragraph context for the statement.
        llm_querier (function): A function that takes a prompt and returns the LLM's response.

    Returns:
        list: List of dictionaries with 'question' and 'answer' keys.
    """
    prompt = {
        "system": (
            "You are an expert at creating question-answer pairs that sharply test the key information at hand. Your task is to generate 1-3 "
            "question-answer pairs that test the knowledge contained in the given statement. "
            "The questions should be clear, specific, and directly testable from the statement. "
            "IMPORTANT: Each answer must be exactly one or two words only. No longer answers. "
            "Return your response as a JSON list of objects with 'question' and 'answer' keys. "
            "Example format: [{'question': 'What algorithm optimizes preferences directly?', 'answer': 'DPO'}]"
        ),
        "user": (
            f"Paragraph context: '{paragraph_context}'\n\n"
            f"Knowledge statement: '{statement}'\n\n"
            f"Generate 1-3 question-answer pairs that test this knowledge. Remember: answers must be 1-2 words only:"
        )
    }
    
    response = llm_querier(prompt)
    
    try:
        import json
        qa_pairs = json.loads(response.strip())
        return qa_pairs if isinstance(qa_pairs, list) else []
    except:
        return []

def generate_qas_for_row(row, max_qas=3):
    """Generate QA pairs for a single row's knowledge statement."""
    statement = row['atomic_knowledge_statement']
    paragraph = row['paragraph']
    
    try:
        qa_pairs = generate_qa_for_statement(
            statement, 
            paragraph,
            lambda prompt: utils.query_llm(prompt, model="gpt-4.1-mini", temperature=0.7)
        )
        return qa_pairs[:max_qas]  # Limit to max_qas
    except Exception as e:
        print(f"Failed to generate QAs for row {row.name}: {e}")
        return []

# --- Main Execution ---

# Initialize tqdm for progress tracking with pandas
tqdm.pandas(desc="Generating QAs")

# Apply the QA generation function to each row of the DataFrame
paper_df_with_probes['knowledge_qa_pairs'] = paper_df_with_probes.progress_apply(
    generate_qas_for_row,
    axis=1
)

# Display the DataFrame with the new column
print(paper_df_with_probes[['atomic_knowledge_statement', 'knowledge_qa_pairs']].head())


### [NOT USED] Creating Ontologies for the Knowledge

In [ ]:
for claim in extracted_claims:
    prompt = {}

    prompt['system'] = """Your task is to act as a strict claim evaluator. You will be given a single claim. Your goal is to determine if this claim meets ALL of the following criteria.

    # Evaluation Criteria

    1.  **Objective and Factual:** Extract statements presented as facts or claims, not opinions, subjective evaluations, or superlative statements.
        *   GOOD: "Direct Preference Optimization (DPO) is an algorithm for aligning language models with human preferences."
        *   BAD: "We believe DPO is the most promising approach for alignment." (Subjective belief)
        *   BAD: "DPO is the best method." (Superlative statement)

    3.  **Generalizable Knowledge, not Paper-Specific Results:** Focus on definitions, mechanisms, and core concepts. Do not extract claims about the paper's specific findings, experimental setup, or citations.
        *   GOOD: "Reinforcement Learning from Human Feedback (RLHF) typically involves training a separate reward model on preference data."
        *   BAD: "Our experiments show a 5% improvement on the benchmark." (Specific result)
        *   BAD: "As shown by Smith et al. (2023), the method is effective." (Relies on a citation)"""

    prompt['user'] = f"""Claim: {claim}. Does this claim strictly meet all four criteria? Respond with only the word "True" or "False"."""
    output = utils.query_llm(prompt, model='gpt-4.1', max_tokens=2)
    print(output)